# Yeni Köklü Corpus Maker — İyileştirilmiş Sürüm

## Hücre 1: Yapılandırma · Yardımcılar · Morfofonemik Kurallar · Osmanlıca Render · Sözlük · Üretici · Test
Tüm düzeltmeler ve yeni özellikler aşağıdaki §-bölümlerinde belgélenmiştir.

In [5]:
import os
import re
import csv
import json
import logging
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
from pathlib import Path
from IPython.display import display, HTML
import zeyrek
import nltk
nltk.download('punkt_tab')
logging.getLogger("zeyrek").setLevel(logging.ERROR)

# ============================================================
# §1  CONFIGURATION
# ============================================================
TEST_SENTENCE = "vefat eden ve yitirilenlere Allah'tan rahmet, kederli ailesine, yakınlarına başsağlığı diliyoruz."

LOOKUP_FILE = "manual_lookup-2.tsv"
ABBREV_FILE = "abbrev_lookup.tsv"
ENABLE_HISTORICAL_ORTHOGRAPHY = True

# ── Batch settings (used in the pipeline cell) ──────────────
INPUT_FILE     = "NLLB_part_15.txt"
OUT_OK         = "fsmtamam.jsonl"
OUT_APPROX     = "fsmyaklasik.jsonl"
OUT_MISSING    = "fsmtamamdegil.jsonl"
MAX_LINES      = None   # 10_000 veya None → unlimited
BATCH_WORKERS  = 4        # ThreadPoolExecutor workers
FLUSH_INTERVAL = 500      # flush output every N processed lines

SCORE_MAP = {
    "exact":    1.0,
    "override": 1.0,
    "surface":  1.0,
    "tags":     1.0,
    "punct":    1.0,
    "english":  0.7,   # new: English transliteration
    "auto":     0.6,
    "missing":  0.0,
}

# ── Vowel sets ───────────────────────────────────────────────
KALIN_UNLULER    = set("aıou")
INCE_UNLULER     = set("eiöü")
YUVARLAK_UNLULER = set("ouöü")
DUZ_UNLULER      = set("aeıi")
_TUM_UNLULER     = KALIN_UNLULER | INCE_UNLULER

# ── Phoneme / punctuation maps ───────────────────────────────
PHONEME_MAP = {
    "a": "ا", "e": "ه", "ı": "ی", "i": "ی",
    "o": "و", "ö": "و", "u": "و", "ü": "و",
    "b": "ب", "c": "ج", "ç": "چ", "d": "د",
    "f": "ف", "g": "گ", "ğ": "غ", "h": "ه",
    "j": "ژ", "l": "ل", "m": "م", "n": "ن",
    "p": "پ", "r": "ر", "ş": "ش", "v": "و",
    "y": "ی", "z": "ز", "k": "ک", "q": "ق",
}

PUNCTUATION_MAP = {",": "،", ";": "؛", "?": "؟", "%": "٪"}

TR_LOWER_MAP     = str.maketrans("IİÇĞÖŞÜ", "ıiçğöşü")
TR_NORMALIZE_MAP = str.maketrans("ÂâÎîÛû",  "AaİiUu")

_TR_WORD_CHARS = r"a-zA-Z0-9_ğüşıöçâîûÂÎÛĞÜŞİÖÇ"
_TR_WORD       = f"[{_TR_WORD_CHARS}]"
_TOKEN_RE      = re.compile(
    rf"{_TR_WORD}+(?:[\u2018\u2019']{_TR_WORD}+)?|[^\s{_TR_WORD_CHARS}]+"
)
_WORD_TOKEN_RE = re.compile(
    rf"^{_TR_WORD}+(?:[\u2018\u2019']{_TR_WORD}+)?$"
)

analyzer = zeyrek.MorphAnalyzer()

# ============================================================
# §2  FSM STATES
#     BUFFER state removed – it had no incoming transitions in
#     TAG_TO_STATE and was therefore unreachable.
# ============================================================
ROOT              = "ROOT"
DERIVATION        = "DERIVATION"
DERIVATION_NOMINAL = "DERIVATION_NOMINAL"
VOICE             = "VOICE"
NEGATION          = "NEGATION"
TENSE             = "TENSE"
PERSON            = "PERSON"
CASE              = "CASE"
PLURAL            = "PLURAL"
POSSESSIVE        = "POSSESSIVE"
COPULA            = "COPULA"

ROOT_POS_TAGS        = {"NOUN", "VERB", "ADJ", "ADV", "PRON", "NUM", "QUES"}
EMPTY_TAGS           = {"NOM", "A3SG"}          # FIX: COPULA removed; it now passes through
PERSON_TAGS          = {"A1SG", "A2SG", "A3SG", "A1PL", "A2PL", "A3PL"}
POSSESSIVE_TAGS      = {"P1SG", "P2SG", "P3SG", "P1PL", "P2PL", "P3PL"}
CASE_TAGS            = {"NOM", "ACC", "DAT", "LOC", "ABL", "GEN", "INS", "REL", "REL_LOC"}
VERBAL_TENSE_TAGS    = {"PAST", "NARR", "PROG", "PROG2", "FUTURE", "FUT_PART", "AOR", "COND", "NECES", "OPT"}
VOICE_TAGS           = {"PASSIVE", "CAUSATIVE", "RECIPROCAL", "REFLEXIVE"}
VERBAL_DERIVATION_TAGS = {
    "ABLE", "UNABLE", "ACQUIRE", "INF1", "INF2", "PART", "PAST_PART", "CONV", "CONV_AFTER", "CONV_BY", "CONV_SINCE", "CONV_ASLONGAS", "CONV_WHILE", "FUT_PART"
}
NOMINAL_DERIVATION_TAGS = {
    "NOM_DER_LIK", "NOM_DER_LI", "NOM_DER_SIZ",
    "NOM_DER_SEL", "NOM_DER_CI", "NOM_DER_DAS", "NOM_DER_MSI",
}

TAG_FSM = {
    ROOT:               [DERIVATION, DERIVATION_NOMINAL, VOICE, NEGATION, TENSE,
                         PERSON, PLURAL, POSSESSIVE, CASE, COPULA],
    DERIVATION:         [DERIVATION, VOICE, NEGATION, TENSE, PLURAL, POSSESSIVE, CASE],
    DERIVATION_NOMINAL: [DERIVATION_NOMINAL, PLURAL, POSSESSIVE, CASE, COPULA],
    VOICE:              [VOICE, DERIVATION, NEGATION, TENSE, PLURAL, POSSESSIVE, CASE],
    NEGATION:           [VOICE, DERIVATION, TENSE],
    TENSE:              [TENSE, PERSON, COPULA, CASE],
    PERSON:             [COPULA, CASE],
    PLURAL:             [POSSESSIVE, CASE, COPULA],
    POSSESSIVE:         [CASE, PLURAL, COPULA],
    CASE:               [CASE, PLURAL, COPULA],
    COPULA:             [TENSE, PERSON, CASE],
}

TAG_TO_STATE = {
    "ABLE":          DERIVATION,
    "UNABLE":        DERIVATION,
    "ACQUIRE":       DERIVATION,
    "INF1":          DERIVATION,
    "INF2":          DERIVATION,
    "PART":          DERIVATION,
    "PAST_PART":     DERIVATION,
    "CONV":          DERIVATION,
    "CONV_AFTER":    DERIVATION,
    "CONV_BY":       DERIVATION,
    "CONV_SINCE":    DERIVATION,
    "CONV_ASLONGAS": DERIVATION,
    "CONV_WHILE":    DERIVATION,
    "PASSIVE":       VOICE,
    "CAUSATIVE":     VOICE,
    "RECIPROCAL":    VOICE,
    "REFLEXIVE":     VOICE,
    "NEG":           NEGATION,
    "PAST":          TENSE,
    "NARR":          TENSE,
    "PROG":          TENSE,
    "PROG2":         TENSE,
    "FUTURE":        TENSE,
    "FUT_PART":      DERIVATION,
    "AOR":           TENSE,
    "COND":          TENSE,
    "NECES":         TENSE,
    "OPT":           TENSE,
    "A1SG":          PERSON,
    "A2SG":          PERSON,
    "A3SG":          PERSON,
    "A1PL":          PERSON,
    "A2PL":          PERSON,
    "A3PL":          PERSON,
    "PLURAL":        PLURAL,
    "P1SG":          POSSESSIVE,
    "P2SG":          POSSESSIVE,
    "P3SG":          POSSESSIVE,
    "P1PL":          POSSESSIVE,
    "P2PL":          POSSESSIVE,
    "P3PL":          POSSESSIVE,
    "NOM":           CASE,
    "ACC":           CASE,
    "DAT":           CASE,
    "LOC":           CASE,
    "ABL":           CASE,
    "GEN":           CASE,
    "INS":           CASE,
    "REL":           CASE,
    "REL_LOC":       CASE,
    "COPULA":        COPULA,
    "COPULA_ASSERT": COPULA,
    "NOM_DER_LIK":   DERIVATION_NOMINAL,
    "NOM_DER_LI":    DERIVATION_NOMINAL,
    "NOM_DER_SIZ":   DERIVATION_NOMINAL,
    "NOM_DER_SEL":   DERIVATION_NOMINAL,
    "NOM_DER_CI":    DERIVATION_NOMINAL,
    "NOM_DER_DAS":   DERIVATION_NOMINAL,
    "NOM_DER_MSI":   DERIVATION_NOMINAL,
}

# ============================================================
# §3  BASIC HELPERS
# ============================================================

def normalize_tr_text(text: str) -> str:
    if not text:
        return text
    return text.translate(TR_NORMALIZE_MAP)


def lower_tr(text: str) -> str:
    if not text:
        return text
    return normalize_tr_text(text).translate(TR_LOWER_MAP).lower()


def fold_tr(text: str) -> str:
    """Accent-fold Turkish text to plain ASCII for fuzzy lookup."""
    if not text:
        return text
    text = lower_tr(text)
    return (
        text.replace("ç", "c")
            .replace("ğ", "g")
            .replace("ı", "i")
            .replace("ö", "o")
            .replace("ş", "s")
            .replace("ü", "u")
    )


def is_word_token(token: str) -> bool:
    return bool(_WORD_TOKEN_RE.fullmatch(token))


def convert_ottoman_punctuation(text: str) -> str:
    return "".join(PUNCTUATION_MAP.get(ch, ch) for ch in text)


def last_vowel(text: str) -> str:
    for ch in reversed(lower_tr(text or "")):
        if ch in _TUM_UNLULER:
            return ch
    return "a"


def is_vowel(ch: str) -> bool:
    return lower_tr(ch or "")[:1] in _TUM_UNLULER


def starts_with_vowel(text: str) -> bool:
    return bool(text) and is_vowel(text[0])


def ends_with_vowel(text: str) -> bool:
    return bool(text) and is_vowel(text[-1])


def get_vowel_type(word: str) -> str:
    return "kalin" if last_vowel(word) in KALIN_UNLULER else "ince"


def choose_harmony_A(root_surface: str) -> str:
    return "a" if last_vowel(root_surface) in KALIN_UNLULER else "e"


def choose_harmony_I(root_surface: str) -> str:
    v = last_vowel(root_surface)
    if v in {"a", "ı"}:
        return "ı"
    if v in {"e", "i"}:
        return "i"
    if v in {"o", "u"}:
        return "u"
    return "ü"


def choose_harmony_U(root_surface: str) -> str:
    return choose_harmony_I(root_surface)


def choose_initial_D(surface: str) -> str:
    if not surface:
        return "d"
    return "t" if lower_tr(surface[-1]) in {"ç", "f", "h", "k", "p", "s", "ş", "t"} else "d"


def choose_initial_C(surface: str) -> str:
    if not surface:
        return "c"
    return "ç" if lower_tr(surface[-1]) in {"ç", "f", "h", "k", "p", "s", "ş", "t"} else "c"


def strip_infinitive_from_ottoman(ot_word: str) -> str:
    for suffix in ("مق", "مك", "ماق", "مەك", "mak", "mek"):
        if ot_word.endswith(suffix):
            return ot_word[: -len(suffix)]
    return ot_word


def normalize_surface_ascii(text: str) -> str:
    text = lower_tr(text)
    text = text.replace("\u2019", "'").replace("\u2018", "'").replace("'", "")
    return text

# ============================================================
# §4  MORPHOPHONEMIC REPRESENTATIONS
# ============================================================

UNDERLYING_MORPHS = {
    "PASSIVE":       "Il",
    "CAUSATIVE":     "DIr",
    "RECIPROCAL":    "Iş",
    "REFLEXIVE":     "In",
    "ABLE":          "(y)Abil",
    "UNABLE":        "mA",
    "ACQUIRE":       "lAn",
    "NEG":           "mA",
    "INF1":          "mAk",
    "INF2":          "mA",
    "PART":          "",
    "PAST_PART":     "DIk",
    "CONV":          "",
    "CONV_AFTER":    "Ip",
    "CONV_BY":       "(y)ArAk",
    "CONV_SINCE":    "(y)AlI",
    "CONV_ASLONGAS": "DIkçA",
    "CONV_WHILE":    "ken",
    "PAST":          "DI",
    "NARR":          "mIş",
    "PROG":          "Iyor",
    "PROG2":         "mAktA",
    "FUTURE":        "(y)AcAk",
    "FUT_PART":      "(y)AcAk",
    "AOR":           "Ar",
    "COND":          "sA",
    "NECES":         "mAlI",
    "OPT":           "(y)A",
    "A1SG":          "Im",
    "A2SG":          "sIn",
    "A3SG":          "",
    "A1PL":          "Iz",
    "A2PL":          "sInIz",
    "A3PL":          "lAr",
    "PLURAL":        "lAr",
    "P1SG":          "(I)m",
    "P2SG":          "(I)n",
    "P3SG":          "sI",
    "P1PL":          "(I)mIz",
    "P2PL":          "(I)nIz",
    "P3PL":          "lArI",
    "NOM":           "",
    "ACC":           "(y)I",
    "DAT":           "(y)A",
    "LOC":           "DA",
    "ABL":           "DAn",
    "GEN":           "(n)In",
    "INS":           "(y)lA",
    "REL":           "ki",
    "REL_LOC":       "DAki",
    "COPULA":        "i",      # bare copula stem; surface realized via copula_mode
    "COPULA_ASSERT": "DIr",
    "NOM_DER_LIK":   "lIk",
    "NOM_DER_LI":    "lI",
    "NOM_DER_SIZ":   "sIz",
    "NOM_DER_SEL":   "sAl",
    "NOM_DER_CI":    "CI",
    "NOM_DER_DAS":   "DAş",
    "NOM_DER_MSI":   "ImsI",
}

# Hand-curated suffix surface → Ottoman script overrides.
# FIX: removed duplicate "leri" key that was present in original.
# NEW: added plural+case, copula+tense, iken/ise, and possessive+case combos.
OTTOMAN_SURFACE_OVERRIDES: dict[str, str] = {
    # ── Plural ──────────────────────────────────────────────────────────
    "lar":       "لر",
    "ler":       "لر",
    # ── Plural + possessive P3 ───────────────────────────────────────────
    "ları":      "لری",
    "leri":      "لری",    # single entry now (duplicate removed)
    # ── Plural + case ────────────────────────────────────────────────────
    "lara":      "لره",
    "lere":      "لره",
    "lardan":    "لردن",
    "lerden":    "لردن",
    "larla":     "لرله",
    "lerle":     "لرله",
    "larda":     "لرده",
    "lerde":     "لرده",
    "ların":     "لرڭ",
    "lerin":     "لرڭ",
    "larını":    "لرینی",
    "lerini":    "لرینی",
    "larına":    "لرینە",
    "lerine":    "لرینە",
    "larında":   "لریندە",
    "lerinde":   "لریندە",
    "larından":  "لریندن",
    "lerinden":  "لریندن",
    "larının":   "لرینڭ",
    "lerinin":   "لرینڭ",
    # ── 1st/2nd person possessive ────────────────────────────────────────
    "ımız":      "مز",
    "imiz":      "مز",
    "umuz":      "مز",
    "ümüz":      "مز",
    "mız":       "مز",
    "miz":       "مز",
    "muz":       "مز",
    "müz":       "مز",
    "ım":        "م",
    "im":        "م",
    "um":        "م",
    "üm":        "م",
    "m":         "م",
    "ın":        "ڭ",
    "in":        "ڭ",
    "un":        "ڭ",
    "ün":        "ڭ",
    "nız":       "ڭز",
    "niz":       "ڭز",
    "nuz":       "ڭز",
    "nüz":       "ڭز",
    "ınız":      "ڭز",
    "iniz":      "ڭز",
    "unuz":      "ڭز",
    "ünüz":      "ڭز",
    # ── 3rd person genitive ──────────────────────────────────────────────
    "nın":       "نڭ",
    "nin":       "نڭ",
    "nun":       "نڭ",
    "nün":       "نڭ",
    # ── P3SG ─────────────────────────────────────────────────────────────
    "sı":        "سی",
    "si":        "سی",
    "su":        "سی",
    "sü":        "سی",
    # ── Accusative ───────────────────────────────────────────────────────
    "ı":         "ی",
    "i":         "ی",
    "u":         "ی",
    "ü":         "ی",
    "yı":        "یی",
    "yi":        "یی",
    "yu":        "یو",
    "yü":        "یو",
    # ── Dative ───────────────────────────────────────────────────────────
    "a":         "ە",
    "e":         "ە",
    "ya":        "یه",
    "ye":        "یە",
    # ── Locative ─────────────────────────────────────────────────────────
    "da":        "ده",
    "de":        "ده",
    "ta":        "ده",
    "te":        "ده",
    # ── Ablative ─────────────────────────────────────────────────────────
    "dan":       "دن",
    "den":       "دن",
    "tan":       "دن",
    "ten":       "دن",
    # ── Instrumental ─────────────────────────────────────────────────────
    "la":        "له",
    "le":        "له",
    "yla":       "یله",
    "yle":       "یله",
    # ── Locative-relative ────────────────────────────────────────────────
    "daki":      "دەكی",
    "deki":      "دەكی",
    "taki":      "دەكی",
    "teki":      "دەكی",
    "ndaki":     "ندەكی",
    "ndeki":     "ندەكی",
    # ── Relative ─────────────────────────────────────────────────────────
    "ki":        "كی",
    # ── Past ─────────────────────────────────────────────────────────────
    "dı":        "دی",
    "di":        "دی",
    "du":        "دی",
    "dü":        "دی",
    "tı":        "دی",
    "ti":        "دی",
    "tu":        "دی",
    "tü":        "دی",
    # ── Copula-assert (dir/tır…) ─────────────────────────────────────────
    "tır":       "در",
    "tir":       "در",
    "tur":       "در",
    "tür":       "در",
    "dır":       "در",
    "dir":       "در",
    "dur":       "در",
    "dür":       "در",
    # ── Narrative past ───────────────────────────────────────────────────
    "mış":       "مش",
    "miş":       "مش",
    "muş":       "مش",
    "müş":       "مش",
    "mamış":     "ممش",
    "memiş":     "ممش",
    # ── Future / future participle ───────────────────────────────────────
    "acak":      "اجق",
    "ecek":      "ەجك",
    "yacak":     "یاجق",
    "yecek":     "یهجك",
    "acağ":      "اجغ",
    "eceğ":      "ەجگ",
    "yacağ":     "یاجغ",
    "yeceğ":     "یهجگ",
    # ── Imperative 2sg ───────────────────────────────────────────────────
    "sın":       "سڭ",
    "sin":       "سڭ",
    "sun":       "سڭ",
    "sün":       "سڭ",
    "sınız":     "سڭز",
    "siniz":     "سڭز",
    "sunuz":     "سڭز",
    "sünüz":     "سڭز",
    # ── Aorist person endings ────────────────────────────────────────────
    "ız":        "ز",
    "iz":        "ز",
    "uz":        "ز",
    "üz":        "ز",
    "ır":        "یر",
    "ir":        "یر",
    "ur":        "یر",
    "ür":        "یر",
    "yız":       "یز",
    "yiz":       "یز",
    "yuz":       "یوز",
    "yüz":       "یوز",
    # ── Aorist / present ─────────────────────────────────────────────────
    "r":         "ر",
    "ar":        "ار",
    "er":        "ر",
    # ── Conditional ──────────────────────────────────────────────────────
    "sa":        "سە",
    "se":        "سە",
    # ── Necessitative ────────────────────────────────────────────────────
    "malı":      "ملی",
    "meli":      "ملی",
    # ── Progressive ──────────────────────────────────────────────────────
    "yor":       "یور",
    "ıyor":      "یور",
    "iyor":      "یور",
    "uyor":      "یور",
    "üyor":      "یور",
    "makta":     "مقدە",
    "mekte":     "مكدە",
    # ── Negation + infinitive combos ─────────────────────────────────────
    "mayalı":    "مایالی",
    "me":        "مە",
    "meyeli":    "میەلی",
    "abil":      "ابیل",
    "ebil":      "ەبیل",
    "ıl":        "یل",
    "il":        "یل",
    "ul":        "ول",
    "ül":        "ول",
    "an":        "ان",
    "en":        "ان",
    "ıp":        "یب",
    "ip":        "یب",
    "up":        "وب",
    "üp":        "وب",
    "dık":       "دق",
    "dik":       "دك",
    "duk":       "دق",
    "dük":       "دك",
    "tık":       "دق",
    "tik":       "دك",
    "tuk":       "دق",
    "tük":       "دك",
    "ıcı":       "یجی",
    "ici":       "یجی",
    "ucu":       "یجی",
    "ücü":       "یجی",
    "arak":      "ارق",
    "erek":      "ارك",
    "lan":       "لان",
    "len":       "لن",
    "lık":       "لق",
    "lik":       "لق",
    "luk":       "لق",
    "lük":       "لق",
    "yın":       "ین",
    "yin":       "ین",
    "yun":       "یون",
    "yün":       "یون",
    "maz":       "مز",
    "mez":       "مز",
    "maman":     "مامن",
    "memen":     "مەمن",
    "mamak":     "مەمق",
    "memek":     "مەمك",
    "yalı":      "یالی",
    "yeli":      "یەلی",
    # ── Copula stem + tense (iken/ise/idi/imiş) ─────────────────────────
    "iken":      "ایکن",
    "yken":      "ایکن",
    "ise":       "ایسه",
    "ysa":       "ایسه",
    "yse":       "ایسه",
    "idi":       "ایدی",
    "ydı":       "یدی",
    "ydi":       "یدی",
    "ydu":       "یدی",
    "ydü":       "یدی",
    "imiş":      "ایمش",
    "ymış":      "ایمش",
    "ymiş":      "ایمش",
    # ── Copula-past + person endings ─────────────────────────────────────
    "ydım":      "ایدم",
    "ydim":      "ایدم",
    "ydın":      "ایدڭ",
    "ydin":      "ایدڭ",
    "ydık":      "ایدق",
    "ydik":      "ایدک",
    "ydınız":    "ایدڭز",
    "ydiniz":    "ایدڭز",
    "ydılar":    "یدیلر",
    "ydiler":    "یدیلر",
    "yalım":     "یالم",
    "yelim":     "یەلم",
    "alım":      "الم",
    "elim":      "الم",
    "mayın":     "مایڭ",
    "meyin":     "مەیڭ",
    # ── Progressive + copula combos ──────────────────────────────────────
    "iyordu":    "ایوردی",
    "iyormuş":   "ایورمش",
    "iyorsa":    "ایورسه",
    "ıyordu":    "ایوردی",
    "ıyormuş":   "ایورمش",
    "ıyorsa":    "ایورسه",
    # ── Ablative + P3 chain ──────────────────────────────────────────────
    "ndan":      "ندن",
    "nden":      "ندن",
    # ── Locative + P3 chain ──────────────────────────────────────────────
    "nda":       "نده",
    "nde":       "نده",
    # ── Dative + P3 chain ────────────────────────────────────────────────
    "na":        "نه",
    "ne":        "نه",
    "ken":       "كن",
    "dıkça":     "دقجە",
    "dikçe":     "دكجە",
    "dukça":     "دقجە",
    "dükçe":     "دكجە",
    "tıkça":     "دقجە",
    "tikçe":     "دكجە",
    "tukça":     "دقجە",
    "tükçe":     "دكجە",
}

VOWEL_DROP_WORDS = {
    "burun", "ağız", "karın", "oğul", "gönül",
    "omuz", "akıl", "şehir", "nehir", "sabır", "ömür",
}

WORD_OVERRIDES: dict[str, str] = {
    "Apple":          "آپپلە",
    "Almanya’da":     "آلمانیەدە",
    "Almanya'da":     "آلمانیەدە",
    "ağır":           "آغير",
    "bağır":          "باغر",
    "binalarının":    "بنالرینڭ",
    "başka":          "باشقە",
    "çabaladığınız":  "چابالادیغڭز",
    "da":             "دە",
    "de":             "دە",
    "del":            "دل",
    "değişikliğe":    "دگیشیكلگە",
    "deneyin":        "دڭیڭ",
    "için":           "ایچون",
    "ile":            "ایلە",
    "lise":           "لیسه",
    "Messi":          "مسّی",
    "oyunda":         "اویوندە",
    "oynama":         "اوينامە",
    "belirtmek":      "بلیرتمك",
    "belirtildi":     "بلیرتیلدی",
    "Şikayetinizi":   "شكایتیڭزی",
    "araştırıp":      "آراشدیریپ",
    "ar":             "آری",
    "azalt":         "آزالت",
    "çalışacağız":    "چالیشاجغز",
    "çözmeye":        "چوزمەیە",
    "başar":          "باشار",
    "ağlayan":        "آغلايان",
    "bulur":          "بولور",
    "bulduğum":       "بولديغم",
    "denizdir":       "دڭزدر",
    "bilgilen":       "بیلگیلن",
    "düş":            "دوش",
    "kazan":          "قزان",
    "umutsuzluğa":    "اوموتسزلغه",
    "yaşa":           "یاشا",
    "yaşam":          "یاشام",
    "yaşayabilmesinin": "یاشایابیلمەسنڭ",
    "yaşayarak":      "یاشایارق",
    "yaşasaydınız":   "یاشاسەیدیڭز",
    "yaşayacaklar":   "یاشایاجقلر",
    "vazgeçmek":      "واز گچمك",
    "vazgeç":         "واز گچ",
    "tesislerinin":   "تأسیسلرینڭ",
    "çatılarına":     "چاتیلارینە",
    "eklemeyi":       "اكلمەیی",
    "yerleştirilecek":"یرلشدیریلەجك",
    "megavat":        "مغه وات",
    "mısınız":        "میسڭز",
    "yalvar":         "یالوار",
    "yetecek":        "یتەجك",
    "zarfında":       "ظرفندە",
    "zorla":          "زورلا",
    "paylaş":         "پایلاش",
    "elektrik":       "الكتریگ",
    "üretecek":       "أورتجك",
    "gelir":          "گلیر",
    "allah":          "اللّٰه",
    "dile":           "دیلە",
    "yakın":          "یاقین",
    "misin":          "میسڭ",
    "beraberliğe":    "برابرلگە",
    "istemişti":      "ایستەمشدی",
    "seyrediyorum":   "سیر ایدییورم",
    "izleyen":        "ایزلەین",
    "gidilen":        "گیدیلن",
    "gelince":        "گلینجە",
    "adamak":         "آدامق",
    "adadı":          "آدادی",
    "meslektaşlarımızı": "مسلكداشلریمزی",
    "hayat":          "حیات",
    "rahmet":         "رحمت",
    "başsağlığı":     "باشساغلغی",
}

# ============================================================
# §5  MORPHOPHONEMIC RULES
#     Fixes:
#       • apply_fusion_rules split into fuse_tag_strings /
#         fuse_realized_morphs; legacy wrapper kept for callers.
#       • possessed flag propagated explicitly through
#         realize_allomorphs instead of fragile surface heuristic.
#       • copula_mode now works correctly because COPULA is no
#         longer filtered by EMPTY_TAGS.
# ============================================================

def apply_vowel_harmony(morph: str, root_surface: str) -> str:
    back_a = choose_harmony_A(root_surface)
    back_i = choose_harmony_I(root_surface)
    return morph.replace("A", back_a).replace("I", back_i)


def apply_vowel_drop(root_surface: str, morph: str) -> str:
    root = lower_tr(root_surface)
    if root not in VOWEL_DROP_WORDS:
        return root_surface
    if not morph or not starts_with_vowel(morph):
        return root_surface
    if len(root_surface) < 3:
        return root_surface
    return root_surface[:-2] + root_surface[-1]


def apply_consonant_softening(root_surface: str, morph: str) -> str:
    if not morph or not starts_with_vowel(morph):
        return root_surface
    lower_root = lower_tr(root_surface)
    if lower_root.endswith("nk"):
        return root_surface[:-1] + "g"
    replacements = {"p": "b", "ç": "c", "t": "d", "k": "ğ"}
    last = lower_root[-1]
    if last in replacements:
        return root_surface[:-1] + replacements[last]
    return root_surface


def apply_buffer_consonants(prev_surface: str, morph: str) -> str:
    for token, char in [("(y)", "y"), ("(n)", "n"), ("(s)", "s"), ("(ş)", "ş")]:
        if morph.startswith(token):
            buf = char if ends_with_vowel(prev_surface) else ""
            return buf + morph[3:]
    return morph


def resolve_nominal_possessive(tag: str, prev_surface: str) -> str:
    hi = choose_harmony_I(prev_surface)
    ha = choose_harmony_A(prev_surface)
    ev = ends_with_vowel(prev_surface)
    if tag == "P1SG":
        return "m"            if ev else hi + "m"
    if tag == "P2SG":
        return "n"            if ev else hi + "n"
    if tag == "P3SG":
        return "s" + hi       if ev else hi
    if tag == "P1PL":
        return "m" + hi + "z" if ev else hi + "m" + hi + "z"
    if tag == "P2PL":
        return "n" + hi + "z" if ev else hi + "n" + hi + "z"
    if tag == "P3PL":
        return "l" + ha + "r" + choose_harmony_I(prev_surface)
    return ""


def resolve_nominal_case(tag: str, prev_surface: str, possessed: bool = False) -> str:
    hi = choose_harmony_I(prev_surface)
    ha = choose_harmony_A(prev_surface)
    ev = ends_with_vowel(prev_surface)
    if tag == "NOM":
        return ""
    if tag == "ACC":
        return ("n" + hi) if possessed else (("y" if ev else "") + hi)
    if tag == "DAT":
        return ("n" + ha) if possessed else (("y" if ev else "") + ha)
    if tag == "LOC":
        return ("n" + choose_initial_D(prev_surface) + ha) if possessed else (choose_initial_D(prev_surface) + ha)
    if tag == "ABL":
        return (("n" + choose_initial_D(prev_surface) + ha + "n") if possessed else (choose_initial_D(prev_surface) + ha + "n"))
    if tag == "GEN":
        return (("n" if ev or possessed else "") + hi + "n")
    if tag == "INS":
        return ("y" if ev else "") + "l" + ha
    if tag == "REL":
        return "ki"
    if tag == "REL_LOC":
        return choose_initial_D(prev_surface) + ha + "ki"
    return ""


def resolve_copula_variant(follow_tag: str, prev_surface: str) -> str:
    """Surface form for copula-stem + tense (büyük-tü, araba-ydı, …)."""
    buf = "y" if ends_with_vowel(prev_surface) else ""
    if follow_tag == "PAST":
        return buf + choose_initial_D(prev_surface) + choose_harmony_I(prev_surface)
    if follow_tag == "NARR":
        return buf + "m" + choose_harmony_I(prev_surface) + "ş"
    if follow_tag == "COND":
        return buf + "s" + choose_harmony_A(prev_surface)
    if follow_tag == "NECES":
        hi = choose_harmony_I(prev_surface)
        return (buf or "i") + "d" + hi + "r"
    return ""


# ── Fusion helpers ─────────────────────────────────────────────────────────

def fuse_tag_strings(tags: list[str]) -> list[str]:
    """Collapse adjacent tag-string pairs (e.g. LOC+REL → REL_LOC)."""
    fused: list[str] = []
    i = 0
    while i < len(tags):
        if tags[i:i + 2] == ["LOC", "REL"]:
            fused.append("REL_LOC")
            i += 2
        else:
            fused.append(tags[i])
            i += 1
    return fused


def fuse_realized_morphs(sequence: list[dict]) -> list[dict]:
    """Adjust realized morph dicts for surface-level assimilation."""
    fused: list[dict] = []
    i = 0
    while i < len(sequence):
        item = dict(sequence[i])
        if fused:
            prev = fused[-1]
            # NEG absorbs following CONV_SINCE / NARR / INF1
            if prev["tag"] == "NEG" and item["tag"] in {"CONV_SINCE", "NARR", "INF1"}:
                prev["tag"] = item["tag"]
                prev["surface"] = prev["surface"] + item["surface"]
                i += 1
                continue
            if prev["tag"] == "COND" and item["tag"] == "PAST":
                item["surface"] = "y" + choose_initial_D(prev["surface"]) + choose_harmony_I(prev["surface"])
            if prev["tag"] == "COND" and item["tag"] in {"A1SG", "A2SG", "A1PL", "A2PL"}:
                hi = choose_harmony_I(prev["surface"])
                item["surface"] = {
                    "A1SG": "m",
                    "A2SG": "n",
                    "A1PL": "k",
                    "A2PL": "n" + hi + "z",
                }[item["tag"]]
            if prev["tag"] == "OPT" and item["tag"] == "A1PL":
                prev["surface"] = prev["surface"] + "l" + choose_harmony_I(prev["surface"]) + "m"
                i += 1
                continue
            if prev["tag"] == "UNABLE" and item["tag"] == "AOR":
                item["surface"] = "z"
            if prev["tag"] == "NEG" and item["tag"] == "AOR":
                prev["tag"] = "AOR"
                prev["surface"] = prev["surface"] + "z"
                i += 1
                continue
            if (prev["tag"] == "NEG" and item["tag"] == "INF2"
                    and i + 1 < len(sequence) and sequence[i + 1]["tag"] == "P2SG"):
                prev["tag"] = "INF2"
                prev["surface"] = prev["surface"] + item["surface"] + sequence[i + 1]["surface"]
                i += 2
                continue
            if prev["tag"] == "INF2" and item["tag"] == "P3SG":
                prev["surface"] = prev["surface"][:-1]
            # FUTURE/FUT_PART: acak → acağ before vowel-initial suffix
            if (prev["tag"] in {"FUTURE", "FUT_PART"}
                    and item["surface"][:1] in _TUM_UNLULER
                    and prev["surface"].endswith(("acak", "ecek", "yacak", "yecek"))):
                prev["surface"] = prev["surface"][:-1] + "ğ"
            if (prev["tag"] == "PAST_PART"
                    and item["surface"][:1] in _TUM_UNLULER
                    and prev["surface"].endswith(("dık", "dik", "duk", "dük", "tık", "tik", "tuk", "tük"))):
                prev["surface"] = prev["surface"][:-1] + "ğ"
            # Past + person endings fuse: geldi+m, geliyor+du+m, eder+di+m ...
            if prev["tag"] == "PAST" and item["tag"] in {"A1SG", "A2SG", "A1PL", "A2PL"}:
                hi = choose_harmony_I(prev["surface"])
                fused_person = {
                    "A1SG": "m",
                    "A2SG": "n",
                    "A1PL": "k",
                    "A2PL": "n" + hi + "z",
                }[item["tag"]]
                item["surface"] = fused_person
            # REL + DAT: ki → ye
            if prev["tag"] == "REL" and item["tag"] == "DAT":
                prev["surface"] = prev["surface"][:-2] + "ye"
                i += 1
                continue
        fused.append(item)
        i += 1
    return fused


def apply_fusion_rules(sequence):
    """Legacy wrapper: dispatches to the appropriate typed fuser."""
    if not sequence:
        return sequence
    if isinstance(sequence[0], str):
        return fuse_tag_strings(sequence)
    return fuse_realized_morphs(sequence)


# ── Underlying morph builder ───────────────────────────────────────────────

def build_underlying_morphs(tags: list[str]) -> list[dict]:
    """Convert normalized tag list to underlying morph dicts.

    COPULA is NO LONGER filtered here; it reaches realize_allomorphs
    so that copula_mode is properly activated for nominal copula forms
    such as 'arabaydı' (araba + COPULA + PAST).
    """
    morphs: list[dict] = []
    for tag in tags:
        if tag in ROOT_POS_TAGS or tag in EMPTY_TAGS:
            continue
        underlying = UNDERLYING_MORPHS.get(tag)
        if underlying is None:
            continue
        morphs.append({"tag": tag, "underlying": underlying})
    return morphs


# ── Allomorph realizer ─────────────────────────────────────────────────────

def realize_single_morph(
    prev_surface: str,
    morph: dict,
    next_tag: str | None = None,
    copula_mode: bool = False,
    possessed: bool = False,   # FIX: explicit flag replaces surface heuristic
) -> str | None:
    tag        = morph["tag"]
    underlying = morph["underlying"]

    if tag in POSSESSIVE_TAGS:
        return resolve_nominal_possessive(tag, prev_surface)
    if tag in CASE_TAGS:
        return resolve_nominal_case(tag, prev_surface, possessed=possessed)
    if tag == "PLURAL":
        return "l" + choose_harmony_A(prev_surface) + "r"

    # Person endings (verbal)
    if tag == "A1SG":
        return choose_harmony_I(prev_surface) + "m"
    if tag == "A1PL":
        return choose_harmony_I(prev_surface) + "z"
    if tag == "A2SG":
        hi = choose_harmony_I(prev_surface)
        return "s" + hi + "n"
    if tag == "A2PL":
        hi = choose_harmony_I(prev_surface)
        return "s" + hi + "n" + hi + "z"
    if tag == "A3PL":
        return "l" + choose_harmony_A(prev_surface) + "r"

    # Copula-mode tense (nominal predicates)
    if copula_mode and tag in {"PAST", "NARR", "COND", "NECES"}:
        return resolve_copula_variant(tag, prev_surface)

    # Verbal tense
    if tag == "PAST":
        return choose_initial_D(prev_surface) + choose_harmony_I(prev_surface)
    if tag == "NARR":
        return "m" + choose_harmony_I(prev_surface) + "ş"
    if tag == "PROG":
        return choose_harmony_I(prev_surface) + "yor"
    if tag == "PROG2":
        ha = choose_harmony_A(prev_surface)
        return "m" + ha + "kt" + ha
    if tag in {"FUTURE", "FUT_PART"}:
        ha  = choose_harmony_A(prev_surface)
        buf = "y" if ends_with_vowel(prev_surface) else ""
        return buf + ha + "c" + ha + "k"
    if tag == "AOR":
        if normalize_surface_ascii(prev_surface) in {"bul"}:
            return choose_harmony_I(prev_surface) + "r"
        if lower_tr(prev_surface).endswith(("dır", "dir", "dur", "dür", "tır", "tir", "tur", "tür")):
            return choose_harmony_I(prev_surface) + "r"
        if ends_with_vowel(prev_surface):
            return "r"
        vowel_count = sum(1 for ch in lower_tr(prev_surface) if ch in _TUM_UNLULER)
        return choose_harmony_I(prev_surface) + "r" if vowel_count > 1 else choose_harmony_A(prev_surface) + "r"
    if tag == "COND":
        return "s" + choose_harmony_A(prev_surface)
    if tag == "NECES":
        return "m" + choose_harmony_A(prev_surface) + "l" + choose_harmony_I(prev_surface)
    if tag == "OPT":
        return ("y" if ends_with_vowel(prev_surface) else "") + choose_harmony_A(prev_surface)

    # Voice / negation / ability
    if tag == "NEG":
        return "m" + choose_harmony_A(prev_surface)
    if tag == "UNABLE":
        if ends_with_vowel(prev_surface):
            ha = choose_harmony_A(prev_surface)
            return "y" + ha + "m" + ha
        return "m" + choose_harmony_A(prev_surface)
    if tag == "ACQUIRE":
        return "l" + choose_harmony_A(prev_surface) + "n"
    if tag == "ABLE":
        return ("y" if ends_with_vowel(prev_surface) else "") + choose_harmony_A(prev_surface) + "bil"
    if tag == "PASSIVE":
        if ends_with_vowel(prev_surface) or lower_tr(prev_surface).endswith("l"):
            return "n"
        return choose_harmony_I(prev_surface) + "l"
    if tag == "CAUSATIVE":
        if ends_with_vowel(prev_surface):
            return "t"
        vowel_count = sum(1 for ch in lower_tr(prev_surface) if ch in _TUM_UNLULER)
        if lower_tr(prev_surface).endswith("l") and vowel_count > 1:
            return "t"
        return choose_initial_D(prev_surface) + choose_harmony_I(prev_surface) + "r"
    if tag == "RECIPROCAL":
        return choose_harmony_I(prev_surface) + "ş"
    if tag == "REFLEXIVE":
        return choose_harmony_I(prev_surface) + "n"

    # Verbal derivation
    if tag in {"PART", "CONV"}:
        return ""
    if tag == "PAST_PART":
        return "d" + choose_harmony_I(prev_surface) + "k"
    if tag == "CONV_AFTER":
        return choose_harmony_U(prev_surface) + "p"
    if tag == "CONV_BY":
        return ("y" if ends_with_vowel(prev_surface) else "") + choose_harmony_A(prev_surface) + "r" + choose_harmony_A(prev_surface) + "k"
    if tag == "INF1":
        return "m" + choose_harmony_A(prev_surface) + "k"
    if tag == "INF2":
        return "m" + choose_harmony_A(prev_surface)
    if tag == "CONV_WHILE":
        return "ken"
    if tag == "CONV_ASLONGAS":
        return choose_initial_D(prev_surface) + choose_harmony_I(prev_surface) + "kç" + choose_harmony_A(prev_surface)

    # Nominal derivation
    if tag == "NOM_DER_LIK":
        return "l" + choose_harmony_I(prev_surface) + "k"
    if tag == "NOM_DER_LI":
        return "l" + choose_harmony_I(prev_surface)
    if tag == "NOM_DER_SIZ":
        return "s" + choose_harmony_I(prev_surface) + "z"
    if tag == "NOM_DER_SEL":
        return "s" + choose_harmony_A(prev_surface) + "l"
    if tag == "NOM_DER_CI":
        if lower_tr(prev_surface).endswith(("t", "d")):
            hi = choose_harmony_I(prev_surface)
            return hi + "c" + hi
        return choose_initial_C(prev_surface) + choose_harmony_I(prev_surface)
    if tag == "NOM_DER_DAS":
        return choose_initial_D(prev_surface) + choose_harmony_A(prev_surface) + "ş"
    if tag == "NOM_DER_MSI":
        return choose_harmony_I(prev_surface) + "ms" + choose_harmony_I(prev_surface)

    # Fallback: apply harmony to underlying abstract morph
    realized = apply_vowel_harmony(underlying, prev_surface)
    return apply_buffer_consonants(prev_surface, realized)


def realize_allomorphs(root_surface: str, morphs: list[dict]) -> tuple[str, list[dict]]:
    """Walk the morph list, compute surface forms, track possessed flag and copula mode."""
    current_root = lower_tr(root_surface)
    realized: list[dict] = []
    copula_mode = False
    has_possessive = False   # FIX: replaces fragile heuristic in realize_single_morph

    for index, morph in enumerate(morphs):
        tag      = morph["tag"]
        next_tag = morphs[index + 1]["tag"] if index + 1 < len(morphs) else None

        # COPULA sets the mode flag and produces no surface form
        if tag == "COPULA":
            copula_mode = True
            continue

        prev_surface = current_root + "".join(p["surface"] for p in realized)
        surface_piece = realize_single_morph(
            prev_surface, morph,
            next_tag=next_tag,
            copula_mode=copula_mode,
            possessed=has_possessive,
        )
        if surface_piece is None:
            continue

        # Apply phonological alternations when first suffix starts with a vowel
        if not realized and starts_with_vowel(surface_piece):
            current_root = apply_vowel_drop(current_root, surface_piece)
            if tag not in {"PASSIVE", "FUTURE", "FUT_PART", "CONV_AFTER", "ABLE", "OPT", "AOR"}:
                current_root = apply_consonant_softening(current_root, surface_piece)
            prev_surface = current_root + "".join(p["surface"] for p in realized)
            surface_piece = realize_single_morph(
                prev_surface, morph,
                next_tag=next_tag,
                copula_mode=copula_mode,
                possessed=has_possessive,
            )

        surface_piece = normalize_surface_ascii(surface_piece)
        realized.append({"tag": tag, "surface": surface_piece})

        # Track possessive for downstream case selection
        if tag in POSSESSIVE_TAGS:
            has_possessive = True

    return current_root, fuse_realized_morphs(realized)

# ============================================================
# §6  OTTOMAN RENDER LAYER
#     FIX: render_ottoman now tracks the *preceding* vowel per
#     character rather than computing one word-level vowel type.
#     This gives correct k/t/s grapheme selection in words whose
#     vowels cross harmony boundaries (loanwords, derived forms).
# ============================================================

def render_ottoman(surface_form: str, historical: bool = False) -> str:
    if not surface_form:
        return ""

    normalized = normalize_surface_ascii(surface_form)
    if historical and normalized in OTTOMAN_SURFACE_OVERRIDES:
        return OTTOMAN_SURFACE_OVERRIDES[normalized]

    result = ""
    # Leading alef for initial vowels
    if normalized and normalized[0] in _TUM_UNLULER:
        result += "ا"

    # FIX: initialise harmony from the FIRST vowel in the word so
    # that consonants preceding any vowel are disambiguated correctly.
    first_v = next((c for c in normalized if c in _TUM_UNLULER), "a")
    current_harmony = "kalin" if first_v in KALIN_UNLULER else "ince"

    for ch in normalized:
        # Update harmony as each new vowel is encountered
        if ch in _TUM_UNLULER:
            current_harmony = "kalin" if ch in KALIN_UNLULER else "ince"

        if ch == "k":
            result += "ق" if current_harmony == "kalin" else "ک"
        elif ch == "t":
            result += "ط" if current_harmony == "kalin" else "ت"
        elif ch == "s":
            result += "ص" if current_harmony == "kalin" else "س"
        else:
            result += PHONEME_MAP.get(ch, ch)

    return result


def merge_ottoman(root_ot: str, realized_suffixes: list[dict]) -> str:
    rendered   = [p["ottoman"] for p in realized_suffixes if p["ottoman"]]
    visible_tags = [p["tag"]   for p in realized_suffixes if p["ottoman"]]
    first_vis  = rendered[0] if rendered else ""

    # ه→ە before plural لر
    if first_vis.startswith("لر") and root_ot.endswith("ه"):
        root_ot = root_ot[:-1] + "ە"
    if first_vis.startswith("لر") and len(rendered) >= 2 and rendered[1] == "ه":
        rendered[1] = "ە"

    # ه→ە before REL_LOC
    if visible_tags[:1] == ["REL_LOC"] and root_ot.endswith("ه"):
        root_ot = root_ot[:-1] + "ە"

    # P3SG + ACC contraction: سی + نی → سنی, ی + نی → ینی
    if (visible_tags[:2] == ["P3SG", "ACC"]
            and rendered[:2] == ["سی", "نی"]):
        if root_ot.endswith("ه"):
            root_ot = root_ot[:-1] + "ە"
        rendered = ["سنی"] + rendered[2:]
    elif (visible_tags[:2] == ["P3SG", "ACC"]
            and rendered[:2] == ["ی", "نی"]):
        rendered = ["ینی"] + rendered[2:]

    # P3SG + DAT contraction: ی + نه → نە
    if (visible_tags[:2] == ["P3SG", "DAT"]
            and rendered[:2] == ["ی", "نه"]):
        rendered = ["نە"] + rendered[2:]

    # P3SG + LOC contraction: ی + نده → نده
    if (visible_tags[:2] == ["P3SG", "LOC"]
            and rendered[:2] == ["ی", "نده"]):
        rendered = ["نده"] + rendered[2:]

    # P3SG + GEN contraction: سی + نڭ → سنڭ
    if (visible_tags[:2] == ["P3SG", "GEN"]
            and rendered[:2] == ["سی", "نڭ"]):
        rendered = ["سنڭ"] + rendered[2:]

    # P3SG + ABL contraction: ی + ندن → ندن
    if (visible_tags[:2] == ["P3SG", "ABL"]
            and rendered[:2] == ["ی", "ندن"]):
        rendered = ["ندن"] + rendered[2:]

    # P2SG + ACC contraction: ن + نی → ڭی
    if (visible_tags[:2] == ["P2SG", "ACC"]
            and rendered[:2] == ["ن", "نی"]):
        if root_ot.endswith("ه"):
            root_ot = root_ot[:-1] + "ە"
        rendered = ["ڭی"] + rendered[2:]

    # P1SG + ACC contraction: م + نی → می
    if (visible_tags[:2] == ["P1SG", "ACC"]
            and rendered[:2] == ["م", "نی"]):
        rendered = ["می"] + rendered[2:]

    # PAST_PART + P2SG + LOC: دیگ + ڭ + نده → دیگندە
    if (visible_tags[:3] == ["PAST_PART", "P2SG", "LOC"]
            and rendered[1:3] == ["ڭ", "نده"]):
        rendered = [rendered[0], "ندە"] + rendered[3:]

    # PAST_PART + P3SG + LOC: دیگ + ی + نده → دیگندە
    if (visible_tags[:3] == ["PAST_PART", "P3SG", "LOC"]
            and rendered[1:3] == ["ی", "نده"]):
        rendered = [rendered[0], "ندە"] + rendered[3:]

    # PAST_PART + P3SG + ACC: دیگ + ی + نی / ینی → دیگنی
    if (visible_tags[:3] == ["PAST_PART", "P3SG", "ACC"]
            and rendered[1:3] == ["ی", "نی"]):
        rendered = [rendered[0], "نی"] + rendered[3:]
    if (visible_tags[:3] == ["PAST_PART", "P3SG", "ACC"]
            and rendered[1:2] == ["ینی"]):
        rendered = [rendered[0], "نی"] + rendered[2:]

    # PAST_PART + P3SG + ABL: دیگ + ی + ندن → دیگندن
    if (visible_tags[:3] == ["PAST_PART", "P3SG", "ABL"]
            and rendered[1:3] == ["ی", "ندن"]):
        rendered = [rendered[0], "ندن"] + rendered[3:]

    for idx in range(len(visible_tags) - 2):
        if (visible_tags[idx:idx + 3] == ["PLURAL", "P3SG", "DAT"]
                and rendered[idx + 1:idx + 3] in (["ی", "نه"], ["ي", "نه"])):
            rendered = rendered[:idx + 1] + ["ینە"] + rendered[idx + 3:]
            break
        if (visible_tags[idx:idx + 3] == ["PAST_PART", "P2SG", "LOC"]
                and rendered[idx + 1:idx + 3] == ["ڭ", "نده"]):
            rendered = rendered[:idx + 1] + ["ندە"] + rendered[idx + 3:]
            break
        if (visible_tags[idx:idx + 3] == ["PAST_PART", "P3SG", "LOC"]
                and rendered[idx + 1:idx + 3] == ["ی", "نده"]):
            rendered = rendered[:idx + 1] + ["ندە"] + rendered[idx + 3:]
            break
        if (visible_tags[idx:idx + 3] == ["PAST_PART", "P3SG", "ACC"]
                and rendered[idx + 1:idx + 3] == ["ی", "نی"]):
            rendered = rendered[:idx + 1] + ["نی"] + rendered[idx + 3:]
            break
        if (visible_tags[idx:idx + 3] == ["PAST_PART", "P3SG", "ABL"]
                and rendered[idx + 1:idx + 3] == ["ی", "ندن"]):
            rendered = rendered[:idx + 1] + ["ندن"] + rendered[idx + 3:]
            break

    # CAUSATIVE + PROG contraction: ت/ط + یور → تییور
    if (visible_tags[:2] == ["CAUSATIVE", "PROG"]
            and rendered[:2] in (["ت", "یور"], ["ط", "یور"])):
        rendered = ["تییور"] + rendered[2:]

    # e-final stems + PROG + A1PL: ...ە + یور + ز → ...ییورز
    if (visible_tags[:2] == ["PROG", "A1PL"]
            and rendered[:2] in (["یور", "ز"], ["يور", "ز"])
            and root_ot.endswith(("ە", "ه"))):
        root_ot = root_ot[:-1] + "ی"

    # ه→ە before ی-initial suffix
    if first_vis.startswith("ی") and root_ot.endswith("ه"):
        root_ot = root_ot[:-1] + "ە"

    return root_ot + "".join(rendered)


def adjust_etmek_auxiliary_output(
    lemma: str | None,
    realized_root: str,
    rendered_suffixes: list[dict],
) -> tuple[str | None, list[dict]]:
    """Normalize generated Ottoman output for auxiliary/finite `etmek` forms.

    The generic verb pipeline strips `ايتمك` to `ايت`, but many productive finite
    forms of `etmek` in this project follow the `اید...` / `ایدی...` pattern
    instead: `eder`, `edecek`, `ediyor`, `edildi`, ...
    """
    if lower_tr(lemma or "") != "etmek":
        return None, rendered_suffixes
    if normalize_surface_ascii(realized_root) != "ed" or not rendered_suffixes:
        return None, rendered_suffixes

    adjusted = [dict(part) for part in rendered_suffixes]
    first_surface = normalize_surface_ascii(adjusted[0].get("surface", ""))
    if not starts_with_vowel(first_surface):
        return None, adjusted

    aux_root_ot = "ایدی" if first_surface.startswith(("iyor", "ıyor", "uyor", "üyor")) else "اید"
    first_ottoman = adjusted[0].get("ottoman", "")
    if first_ottoman.startswith("ه"):
        adjusted[0]["ottoman"] = "ە" + first_ottoman[1:]
    return aux_root_ot, adjusted


def adjust_past_person_rendering(rendered_suffixes: list[dict]) -> list[dict]:
    """Render fused past+person endings as دم/دڭ/... instead of دیم/دین/..."""
    adjusted = [dict(part) for part in rendered_suffixes]
    for i in range(1, len(adjusted)):
        prev = adjusted[i - 1]
        item = adjusted[i]
        if prev.get("tag") != "PAST":
            continue
        if item.get("tag") not in {"A1SG", "A2SG", "A1PL", "A2PL"}:
            continue
        prev_surface = normalize_surface_ascii(prev.get("surface", ""))
        starts_with_yd = prev_surface.startswith("yd")
        prev["ottoman"] = "ید" if starts_with_yd else "د"
        if item.get("tag") == "A2SG":
            item["ottoman"] = "ڭ"
        elif item.get("tag") == "A1PL":
            item["ottoman"] = "ق" if starts_with_yd else "ك"
        elif item.get("tag") == "A2PL":
            prev["ottoman"] = "یدی" if starts_with_yd else "دی"
            item["ottoman"] = "ڭز"
    return adjusted


def adjust_softened_nominal_root_ottoman(root_ot: str, lemma: str, realized_root: str) -> str:
    base = normalize_surface_ascii(lemma or "")
    realized = normalize_surface_ascii(realized_root or "")
    if not base or not realized or len(base) != len(realized):
        return root_ot
    if base[:-1] != realized[:-1]:
        return root_ot
    pair = (base[-1], realized[-1])
    if pair == ("p", "b"):
        if root_ot.endswith("پ"):
            return root_ot[:-1] + "ب"
        return root_ot
    if pair == ("ç", "c") and root_ot.endswith("چ"):
        return root_ot[:-1] + "ج"
    if pair == ("t", "d") and root_ot.endswith(("ت", "ط")):
        return root_ot[:-1] + "د"
    if pair == ("k", "ğ"):
        if root_ot.endswith("ق"):
            return root_ot[:-1] + "غ"
        if root_ot.endswith(("ک", "ك")):
            return root_ot[:-1] + "گ"
    return root_ot


# ============================================================
# §7  ENGLISH TRANSLITERATION
#     Common Ottoman-era practice for transcribing European
#     (primarily English) words into Arabic script.
#
#     Strategy
#     ─────────
#     1. Scan left→right trying the longest matching grapheme
#        pattern first (trigraphs before digraphs before singles).
#     2. Apply context-sensitive rules: soft-c/g, silent final-e,
#        silent initial clusters (kn, wr, gn, ps), -ght reduction.
#     3. Add an initial alef (ا) for vowel-initial words.
#     4. Use ڤ for 'v', پ for 'p' and و for 'w' – marks standard
#        in late Ottoman borrowing conventions.
# ============================================================

_ENG_VOWELS = set("aeiou")

# Ordered patterns: tried longest-first at each position.
# Each entry: (grapheme, ottoman_equivalent)
_ENG_PATTERNS: list[tuple[str, str]] = [
    # ── 4-char ────────────────────────────────────────────
    ("tion",  "شن"),    # nation, station
    ("sion",  "شن"),    # mission; vision → زن handled below
    ("ture",  "چر"),    # future, nature
    ("ough",  "و"),     # though, dough (most common reading)
    ("augh",  "اف"),    # laugh, caught → use sound value
    ("ight",  "ایت"),   # light, night, right
    # ── 3-char ────────────────────────────────────────────
    ("tch",   "چ"),     # watch, catch, match
    ("dge",   "ج"),     # judge, bridge, lodge
    ("sch",   "ش"),     # school, schedule
    ("ght",   "ت"),     # bought, thought (after augh/ough already consumed)
    ("gue",   "گ"),     # league, vague,ogue
    ("que",   "ک"),     # unique, opaque
    ("igh",   "ای"),    # high, sigh, thigh
    ("ssi",   "ش"),     # mission, passion
    ("sci",   "ش"),     # conscious, science
    ("kno",   "نو"),    # know, knowledge
    ("wri",   "ری"),    # write, wrong
    ("psy",   "سی"),    # psychology
    ("pneu",  "نو"),    # pneumonia
    ("rhy",   "ری"),    # rhythm
    # ── 2-char ────────────────────────────────────────────
    ("ph",    "ف"),     # phone, graph, phd
    ("sh",    "ش"),     # show, wish, cash
    ("ch",    "چ"),     # chair, church, much
    ("th",    "ث"),     # the, think (ث for both voiced/voiceless)
    ("wh",    "و"),     # what, where, why
    ("ck",    "ک"),     # back, black, rock
    ("ng",    "ڭ"),     # ring, sing, long
    ("nk",    "ڭک"),   # bank, think, rank
    ("qu",    "کو"),    # queen, quick, quote
    ("kn",    "ن"),     # knife, know, kneel (initial silent k)
    ("wr",    "ر"),     # write, wrong, wrap (initial silent w)
    ("gn",    "ن"),     # gnome, gnaw (initial silent g)
    ("ps",    "س"),     # psychology, psalm (initial silent p)
    ("mb",    "م"),     # lamb, thumb, bomb (final silent b)
    ("gh",    ""),      # though, through, high → silent (default)
    ("ee",    "ی"),     # see, feel, tree
    ("ea",    "ی"),     # eat, tea, read (most common)
    ("oo",    "و"),     # food, moon, cool
    ("ou",    "او"),    # out, about, found
    ("ow",    "او"),    # show, snow / cow, now (default: او)
    ("oa",    "و"),     # boat, coat, road
    ("ai",    "ای"),    # rain, pain, wait
    ("ay",    "ای"),    # day, play, say
    ("ei",    "ی"),     # receive, ceiling, veil
    ("ie",    "ی"),     # field, believe, piece
    ("oi",    "وی"),    # coin, voice, join
    ("oy",    "وی"),    # boy, toy, annoy
    ("au",    "او"),    # auto, cause, haul
    ("aw",    "او"),    # law, saw, draw
    ("ew",    "یو"),    # new, few, dew
    ("ue",    "یو"),    # blue, true, clue
    ("oa",    "و"),     # boat (duplicate guard – dict order wins)
    ("ae",    "ی"),     # aesthetic, algae
    ("oe",    "و"),     # foe, toe, shoe
    ("ui",    "وی"),    # suit, fruit, build
    ("ia",    "یه"),    # piano, tiara, media
    ("io",    "یو"),    # radio, studio
    ("ua",    "وه"),    # usual, gradual
    ("lk",    "لک"),    # milk, silk, bulk
    ("lm",    "لم"),    # film, helm, calm
    ("mn",    "م"),     # autumn, column (final silent n)
    ("rh",    "ر"),     # rhythm, rhyme
    ("xc",    "کس"),    # excel, excess
    ("ww",    "و"),     # www
    ("ss",    "س"),     # class, address (double → single)
    ("ll",    "ل"),     # ball, hill, call (double → single)
    ("tt",    "ت"),     # butter, bitter (double → single)
    ("nn",    "ن"),     # inn, announce
    ("rr",    "ر"),     # error, mirror
    ("pp",    "پ"),     # apple, happen
    ("bb",    "ب"),     # abbey, rubber
    ("dd",    "د"),     # add, odd
    ("ff",    "ف"),     # off, staff
    ("gg",    "گ"),     # egg, suggest
    ("cc",    "ک"),     # account, occur
    ("mm",    "م"),     # common, summer
    ("zz",    "ز"),     # buzz, jazz
]

# Single-character fallback map (English grapheme → Ottoman)
_ENG_SINGLE: dict[str, str] = {
    "a":  "ه",   # short a: cat, hat
    "b":  "ب",
    "c":  "ک",   # hard-c default; soft-c handled contextually
    "d":  "د",
    "e":  "ه",   # short e; terminal silent-e handled contextually
    "f":  "ف",
    "g":  "گ",   # hard-g default; soft-g handled contextually
    "h":  "ه",
    "i":  "ی",
    "j":  "ج",
    "k":  "ک",
    "l":  "ل",
    "m":  "م",
    "n":  "ن",
    "o":  "و",
    "p":  "پ",
    "q":  "ق",
    "r":  "ر",
    "s":  "س",
    "t":  "ت",
    "u":  "ا",   # short u: cut, but, run
    "v":  "ڤ",
    "w":  "و",
    "x":  "کس",
    "y":  "ی",
    "z":  "ز",
}

# Words whose English-like orthography is best handled entirely by
# a hand-written Ottoman form; checked before the grapheme engine.
ENGLISH_WORD_OVERRIDES: dict[str, str] = {
    "internet":   "اینترنت",
    "computer":   "کمپیوتر",
    "digital":    "دیجیتل",
    "software":   "سافتویر",
    "hardware":   "هاردویر",
    "network":    "نتورک",
    "telephone":  "تلفون",
    "television": "تلویزیون",
    "radio":      "رادیو",
    "video":      "ویدیو",
    "photo":      "فوطو",
    "camera":     "کامره",
    "email":      "ایمیل",
    "website":    "ویب سایت",
    "password":   "پاسورد",
    "download":   "داونلود",
    "upload":     "آپلود",
    "london":     "لندره",
    "paris":      "پاریس",
    "berlin":     "برلین",
    "moscow":     "مسکو",
    "new york":   "نیویورک",
    "ok":         "اوکی",
    "okay":       "اوکی",
    "yes":        "یس",
    "no":         "نو",
    "hello":      "هللو",
    "bye":        "بای",
    "english":    "انگیلیزچه",
    "french":     "فرانسزچه",
    "german":     "الماندجه",
    "america":    "امریقا",
    "europe":     "اوروپا",
    "asia":       "آسیا",
    "africa":     "افریقا",
}


def is_likely_english(word: str) -> bool:
    """Heuristic: does this token look like an English/Latin-script foreign word?"""
    w = word.lower()
    # Turkish-specific characters → definitely not English
    if any(c in w for c in "ğüşıöç"):
        return False
    # Strong markers
    if any(pat in w for pat in ("wh", "ph", "tch", "tion", "ght", "ough", "kn", "wr")):
        return True
    if "w" in w or "x" in w:
        return True
    # Common English suffixes
    for sfx in ("tion", "sion", "ness", "ment", "ful", "less", "ive",
                "ous", "ing", "ance", "ence", "able", "ible", "ity",
                "ify", "ize", "ise", "ism", "ist", "ish", "ward"):
        if w.endswith(sfx) and len(w) > len(sfx) + 1:
            return True
    return False


def render_english_ottoman(word: str) -> str:
    """Transliterate an English word into Ottoman Arabic script.

    Applies Ottoman-era borrowing conventions:
      - Digraph-first scanning (longest match wins)
      - Context-sensitive soft-c / soft-g
      - Silent final-e elision
      - Initial alef for vowel-initial pronunciation
      - ڤ for v, پ for p, و for w
    """
    w_lower = word.lower()

    # Hand-written overrides first
    if w_lower in ENGLISH_WORD_OVERRIDES:
        return ENGLISH_WORD_OVERRIDES[w_lower]

    s   = w_lower
    n   = len(s)
    out = ""
    i   = 0

    while i < n:
        matched = False

        # Try patterns longest-first
        for pattern, ottoman in _ENG_PATTERNS:
            plen = len(pattern)
            if i + plen <= n and s[i:i + plen] == pattern:
                # Context-sensitive overrides on matched patterns
                if pattern == "gh" and i == 0:
                    ottoman = "غ"          # ghost, ghee: initial gh is sounded
                elif pattern == "mb" and i + plen < n:
                    ottoman = "مب"         # not word-final → both letters sounded
                elif pattern == "ou" and i + plen < n and s[i + plen] in "lr":
                    ottoman = "ور"         #our → ور (colour, hour)
                elif pattern == "ow" and i + plen < n and s[i + plen] not in _ENG_VOWELS:
                    pass                   # keep "او"
                out += ottoman
                i   += plen
                matched = True
                break

        if matched:
            continue

        ch = s[i]

        # Context-sensitive single-character rules
        if ch == "c":
            if i + 1 < n and s[i + 1] in "eiy":
                out += "س"               # soft c: city, cent, cycle
            else:
                out += "ک"               # hard c: cat, cool
        elif ch == "g":
            if i + 1 < n and s[i + 1] in "eiy":
                out += "ج"               # soft g: gem, ginger, gym
            else:
                out += "گ"               # hard g: go, got, gun
        elif ch == "e" and i == n - 1 and i > 0 and s[i - 1] not in _ENG_VOWELS:
            pass                         # terminal silent-e: name, hate, like
        elif ch == "e" and i == n - 1 and n > 2 and s[n - 3:n - 1] in (
                "bl", "gl", "kl", "pl", "sl", "fl", "cl", "zl"):
            pass                         # silent e after consonant cluster: table, apple
        elif ch == "y":
            if i == 0 or s[i - 1] not in _ENG_VOWELS:
                out += "ی"               # consonantal y: yes, year
            else:
                out += "ی"               # vocalic y: day, joy
        elif ch == "a":
            # Check whether this 'a' is followed by a silent-e pattern (VCe)
            if (i + 2 < n
                    and s[i + 1] not in _ENG_VOWELS
                    and s[i + 2] == "e"
                    and i + 3 >= n):
                out += "ای"              # name, late, cake → long-a reading
            else:
                out += _ENG_SINGLE.get(ch, ch)
        elif ch == "i":
            if (i + 2 < n
                    and s[i + 1] not in _ENG_VOWELS
                    and s[i + 2] == "e"
                    and i + 3 >= n):
                out += "ای"              # bike, like, mine → long-i
            else:
                out += "ی"
        elif ch == "o":
            if (i + 2 < n
                    and s[i + 1] not in _ENG_VOWELS
                    and s[i + 2] == "e"
                    and i + 3 >= n):
                out += "و"               # hope, note, code → long-o
            else:
                out += "و"
        else:
            out += _ENG_SINGLE.get(ch, ch)

        i += 1

    # Prepend alef if the rendered word starts with a vowel glyph
    vowel_glyphs = {"ه", "ی", "و", "ا", "او", "ای"}
    if out and out[0] in {"ه", "ی", "و"}:
        out = "ا" + out

    return out

# ============================================================
# §8  DICTIONARY + ZEYREK ANALYSIS
#     Fixes:
#       • lru_cache maxsize raised to 8192
#       • apostrophe suffix handling extended to ALL case/possessive
#         suffixes, not just the four genitive forms
#       • score_parse: derivation tags excluded from the length
#         penalty to remove double-counting
# ============================================================

def load_tsv_dict(filepath: str) -> dict:
    data: dict = {}
    if not os.path.exists(filepath):
        return data
    with open(filepath, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f, delimiter="\t")
        for row in reader:
            if len(row) >= 2 and not row[0].startswith("#"):
                key = row[0].strip()
                val = row[1].strip()
                if key.lower() == "word" and val.lower() == "ottoman":
                    continue
                data[key] = val
                data[lower_tr(key)] = val
    return data


lookup_dict  = load_tsv_dict(LOOKUP_FILE)
abbrev_dict  = load_tsv_dict(ABBREV_FILE)
lookup_folded: dict = {}
for _k, _v in lookup_dict.items():
    _f = fold_tr(_k)
    if _f and _f not in lookup_folded:
        lookup_folded[_f] = _v


@lru_cache(maxsize=8192)   # FIX: raised from 1000
def cached_analyze(word: str):
    return analyzer.analyze(normalize_tr_text(word))


def flatten_analyses(word: str) -> list:
    return [parse for group in cached_analyze(word) for parse in group]


def parse_header_pos(parse) -> str:
    fmt   = getattr(parse, "formatted", "") or ""
    match = re.match(r"\[([^:]+):([^\],]+)", fmt)
    return match.group(2).strip().upper() if match else str(getattr(parse, "pos", "")).upper()


def extract_surface_root(parse) -> str:
    fmt = getattr(parse, "formatted", "") or ""
    if "] " in fmt:
        tail    = fmt.split("] ", 1)[1]
        segment = re.split(r"[+|]", tail)[0]
        if ":" in segment:
            return lower_tr(segment.split(":", 1)[0].strip())
    lemma = lower_tr(getattr(parse, "lemma", "") or "")
    if lemma.endswith(("mak", "mek")):
        return lemma[:-3]
    return lemma


def lookup_root_entry(key: str):
    if not key:
        return None
    for candidate, score in [
        (key,           32),
        (lower_tr(key), 28),
    ]:
        if candidate in WORD_OVERRIDES:
            return WORD_OVERRIDES[candidate], candidate, score
    for candidate, score in [
        (key,           30),
        (lower_tr(key), 24),
    ]:
        if candidate in lookup_dict:
            return lookup_dict[candidate], candidate, score
    folded = fold_tr(key)
    if folded in lookup_folded:
        return lookup_folded[folded], lower_tr(key), 18
    return None


SURFACE_VERB_FALLBACK_SUFFIXES: dict[str, str] = {
    "irken":  "یركن",
    "rken":   "ركن",
    "sınız":  "سینز",
    "siniz":  "سینز",
    "sunuz":  "سینز",
    "sünüz":  "سینز",
}

SURFACE_FUTURE_CHAIN_SUFFIXES: dict[str, str] = {
    "tim":    "دم",
    "tin":    "دڭ",
    "ti":     "دی",
    "tık":    "دق",
    "tik":    "دك",
    "tuk":    "دق",
    "tük":    "دك",
    "tiniz":  "دیڭز",
    "tiler":  "دیلر",
    "lar":    "لر",
    "ler":    "لر",
    "mış":    "مش",
    "miş":    "مش",
    "muş":    "مش",
    "müş":    "مش",
}


def generate_surface_verb_fallback(token: str):
    word = lower_tr(token)
    for suffix, suffix_ottoman in sorted(
        SURFACE_VERB_FALLBACK_SUFFIXES.items(),
        key=lambda item: len(item[0]),
        reverse=True,
    ):
        if not word.endswith(suffix) or len(word) <= len(suffix):
            continue
        stem = word[:-len(suffix)]
        root_entry = lookup_root_entry(stem)
        selected_stem = select_parse_for_generation(stem)
        if not root_entry and not selected_stem:
            continue
        if not any(parse_header_pos(parse) == "VERB" for parse in flatten_analyses(stem)):
            continue
        if root_entry:
            root_ot = strip_infinitive_from_ottoman(root_entry[0])
        else:
            root_ot = strip_infinitive_from_ottoman(selected_stem["root_ot"])
        return root_ot + suffix_ottoman, stem, suffix
    return None


def generate_surface_future_chain_fallback(token: str):
    word = lower_tr(token)
    future_bases = ("acak", "ecek", "yacak", "yecek")
    for suffix, suffix_ottoman in sorted(
        SURFACE_FUTURE_CHAIN_SUFFIXES.items(),
        key=lambda item: len(item[0]),
        reverse=True,
    ):
        if not word.endswith(suffix) or len(word) <= len(suffix):
            continue
        future_form = word[:-len(suffix)]
        if not future_form.endswith(future_bases):
            continue
        future_ottoman, future_source, _future_dbg = transliterate_token(future_form)
        if future_source == "missing":
            continue
        return future_ottoman + suffix_ottoman, future_form, suffix
    return None


def resolve_root_from_parse(parse):
    lemma = getattr(parse, "lemma", None)
    if not lemma or lemma == "Unk":
        return None
    base_pos     = parse_header_pos(parse)
    surface_root = extract_surface_root(parse)
    candidates   = []
    if base_pos == "VERB" and surface_root:
        candidates.append(surface_root)
    candidates.extend([lemma, lower_tr(lemma)])
    if base_pos == "VERB" and lower_tr(lemma).endswith(("mak", "mek")):
        candidates.append(lower_tr(lemma)[:-3])
    if surface_root and surface_root not in candidates:
        candidates.append(surface_root)

    seen: set = set()
    for candidate in candidates:
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)
        found = lookup_root_entry(candidate)
        if found:
            root_ot, dict_form, lookup_score = found
            return {
                "root_ot":      root_ot,
                "lemma":        lemma,
                "dict_form":    dict_form,
                "lookup_score": lookup_score,
                "base_pos":     base_pos,
                "surface_root": surface_root or lower_tr(candidate),
            }
    return None


ZEYREK_TAG_MAP: dict[str, str] = {
    "Passive":      "PASSIVE",
    "Caus":         "CAUSATIVE",
    "Recip":        "RECIPROCAL",
    "Reflex":       "REFLEXIVE",
    "Able":         "ABLE",
    "Unable":       "UNABLE",
    "Acquire":      "ACQUIRE",
    "Neg":          "NEG",
    "Inf1":         "INF1",
    "Inf2":         "INF2",
    "Part":         "PART",
    "PastPart":     "PAST_PART",
    "FutPart":      "FUT_PART",
    "Conv":         "CONV",
    "AfterDoingSo": "CONV_AFTER",
    "ByDoingSo":    "CONV_BY",
    "SinceDoingSo": "CONV_SINCE",
    "AsLongAs":    "CONV_ASLONGAS",
    "While":        "CONV_WHILE",
    "Past":         "PAST",
    "Narr":         "NARR",
    "Pass":         "PASSIVE",
    "Prog1":        "PROG",
    "Prog2":        "PROG2",
    "Fut":          "FUTURE",
    "Aor":          "AOR",
    "Cond":         "COND",
    "Desr":         "COND",
    "Neces":        "NECES",
    "Opt":          "OPT",
    "Cop":          "COPULA_ASSERT",
    "A1sg":         "A1SG",
    "A2sg":         "A2SG",
    "A3sg":         "A3SG",
    "A1pl":         "A1PL",
    "A2pl":         "A2PL",
    "A3pl":         "A3PL",
    "P1sg":         "P1SG",
    "P2sg":         "P2SG",
    "P3sg":         "P3SG",
    "P1pl":         "P1PL",
    "P2pl":         "P2PL",
    "P3pl":         "P3PL",
    "Nom":          "NOM",
    "Acc":          "ACC",
    "Dat":          "DAT",
    "Loc":          "LOC",
    "Abl":          "ABL",
    "Gen":          "GEN",
    "Ins":          "INS",
    "Rel":          "REL",
    "With":         "NOM_DER_LI",
    "Without":      "NOM_DER_SIZ",
    "Agt":          "NOM_DER_CI",
    "Ness":         "NOM_DER_LIK",
    "FitFor":       "NOM_DER_LIK",
}


def zeyrek_parse_to_tags(parse) -> list[str]:
    base_pos        = parse_header_pos(parse)
    morphemes       = list(getattr(parse, "morphemes", []) or [])
    tags            = [base_pos]
    nominal_context = base_pos in {"NOUN", "ADJ", "PRON", "NUM", "QUES"}
    verbal_context  = base_pos == "VERB"
    prev_raw        = morphemes[0] if morphemes else None
    surface_root    = extract_surface_root(parse)

    for index, raw in enumerate(morphemes[1:], start=1):
        next_raw = morphemes[index + 1] if index + 1 < len(morphemes) else None
        if raw == "Zero":
            prev_raw = raw
            continue
        if raw in {"Noun", "Adj", "Adv", "Pron", "Num"}:
            nominal_context = True
            verbal_context  = False
            prev_raw = raw
            continue
        if raw == "Verb":
            if not verbal_context and prev_raw == "Zero":
                tags.append("COPULA")
            elif not verbal_context:
                tags[0] = "VERB"
            verbal_context  = True
            nominal_context = False
            prev_raw = raw
            continue
        if raw == "A3pl":
            if nominal_context and next_raw == "Become":
                prev_raw = raw
                continue
            tags.append("PLURAL" if nominal_context else "A3PL")
            prev_raw = raw
            continue
        if raw == "A3sg" and nominal_context and next_raw == "Become":
            prev_raw = raw
            continue
        if raw == "A3sg" and not nominal_context:
            tags.append("A3SG")
            prev_raw = raw
            continue
        if raw == "Recip" and surface_root.endswith("ş"):
            prev_raw = raw
            continue
        normalized = ZEYREK_TAG_MAP.get(raw)
        if normalized:
            tags.append(normalized)
        prev_raw = raw

    return fuse_tag_strings(tags)


def validate_question_chain(tags: list[str]) -> bool:
    return all(t in PERSON_TAGS | {"A3SG"} for t in tags[1:])


def validate_nominal_chain(tags: list[str]) -> bool:
    seen_copula = False
    for tag in tags[1:]:
        if tag == "COPULA":
            seen_copula = True
            continue
        if tag in VOICE_TAGS or tag in {"NEG", "PROG", "FUTURE", "AOR", "OPT", "ABLE", "UNABLE", "CONV_AFTER", "CONV_BY", "CONV_SINCE", "CONV_ASLONGAS", "CONV_WHILE"}:
            return False
        if tag in {"PAST", "NARR", "COND", "NECES"} and not seen_copula:
            return False
        if tag in EMPTY_TAGS:
            continue
        if tag in PERSON_TAGS and not seen_copula and tag not in {"A3SG", "A3PL"}:
            return False
    return True


def validate_verbal_chain(tags: list[str]) -> bool:
    nominalized = False
    for tag in tags[1:]:
        if tag in {"INF1", "INF2", "PART", "PAST_PART", "CONV", "CONV_AFTER", "CONV_BY", "CONV_SINCE", "CONV_ASLONGAS", "CONV_WHILE", "FUT_PART"}:
            nominalized = True
            continue
        if tag in NOMINAL_DERIVATION_TAGS:
            nominalized = True
            continue
        if (tag in POSSESSIVE_TAGS | CASE_TAGS | {"PLURAL"}
                and not nominalized
                and "COPULA" not in tags
                and tag != "A3PL"):
            return False
    return True


def validate_tag_sequence(tags: list[str]) -> bool:
    root_pos = tags[0] if tags else ROOT
    if root_pos == "QUES" and not validate_question_chain(tags):
        return False
    if root_pos in {"NOUN", "ADJ", "PRON", "NUM"} and not validate_nominal_chain(tags):
        return False
    if root_pos == "VERB" and not validate_verbal_chain(tags):
        return False

    current_state = ROOT
    for tag in tags:
        if tag in ROOT_POS_TAGS or tag in EMPTY_TAGS:
            continue
        target_state = TAG_TO_STATE.get(tag)
        if target_state is None:
            return False
        if target_state != current_state and target_state not in TAG_FSM.get(current_state, []):
            return False
        current_state = target_state
    return True


def score_parse(parse, normalized_tags: list[str], root_info: dict, ambiguity_count: int) -> float:
    score = 0.0
    score += root_info["lookup_score"]

    if validate_tag_sequence(normalized_tags):
        score += 40
    else:
        score -= 80

    base_pos   = root_info["base_pos"]
    actual_pos = str(getattr(parse, "pos", "")).upper()
    if base_pos == actual_pos:
        score += 10
    elif base_pos == "VERB" and actual_pos in {"ADJ", "ADV"} and len(normalized_tags) == 1:
        # Penalize analyses whose output POS is lost during tag normalization.
        score -= 24
    if "PROP" in (getattr(parse, "formatted", "") or "").upper():
        score -= 20

    # FIX: derivation tags are excluded from the length-penalty window to
    # avoid double-counting (they already carry their own derivation_penalty).
    derivation_penalty = sum(
        1 for t in normalized_tags
        if t in VERBAL_DERIVATION_TAGS or t in NOMINAL_DERIVATION_TAGS
    )
    non_derivation_tags = [
        t for t in normalized_tags
        if t not in VERBAL_DERIVATION_TAGS and t not in NOMINAL_DERIVATION_TAGS
    ]
    score -= derivation_penalty * 4
    score -= max(len(non_derivation_tags) - 3, 0)      # FIX: only non-derivation count
    score -= max(ambiguity_count - 1, 0) * 2

    surface_word = lower_tr(getattr(parse, "word", "") or "")
    formatted    = getattr(parse, "formatted", "") or ""
    morphemes     = list(getattr(parse, "morphemes", []) or [])
    if surface_word.endswith(("sa", "se")):
        score += 6 if "COND" in normalized_tags else -6
    if surface_word.endswith(("mış", "miş", "muş", "müş")):
        score += 10 if "NARR" in normalized_tags else -10
    if "Dim" in morphemes and surface_word.endswith(("ceğiz", "cağız", "ceğim", "cağım", "ceksin", "caksın", "ceksiniz", "caksınız")):
        score -= 24
    if ("Fut" in morphemes and any(tag in normalized_tags for tag in {"A1SG", "A1PL", "A2SG", "A2PL"})
            and surface_word.endswith(("ceğim", "cağım", "ceğiz", "cağız", "ceksin", "caksın", "ceksiniz", "caksınız"))):
        score += 18
    if "ByDoingSo" in morphemes:
        score += 32 if root_info["base_pos"] == "VERB" else -32
    if "Recip" in morphemes and root_info["surface_root"].endswith("ş"):
        score += 10
    if normalized_tags[-2:] == ["P3SG", "DAT"] and surface_word.endswith(("ına", "ine", "una", "üne")):
        score += 1
    if normalized_tags[-2:] == ["P3SG", "ACC"] and surface_word.endswith(("ını", "ini", "unu", "ünü")):
        score += 1
    if normalized_tags[-2:] == ["P3SG", "ABL"] and surface_word.endswith(("ından", "inden", "undan", "ünden")):
        score += 1
    if normalized_tags[-2:] == ["P3SG", "LOC"] and surface_word.endswith(("ında", "inde", "unda", "ünde", "nda", "nde")):
        score += 1
    if normalized_tags[-2:] == ["P3SG", "GEN"] and surface_word.endswith(("ının", "inin", "unun", "ünün")):
        score += 1

    if root_info["dict_form"] == lower_tr(getattr(parse, "lemma", "") or ""):
        score += 6
    return score


def select_parse_for_generation(word: str):
    parses     = flatten_analyses(word)
    candidates = []
    for parse in parses:
        root_info = resolve_root_from_parse(parse)
        if not root_info:
            continue
        ntags = zeyrek_parse_to_tags(parse)
        candidates.append((
            score_parse(parse, ntags, root_info, len(parses)),
            parse, root_info, ntags,
        ))
    if not candidates:
        return None
    candidates.sort(
        key=lambda item: (item[0], -len(item[3]), len(item[2]["surface_root"])),
        reverse=True,
    )
    best_score, best_parse, root_info, tags = candidates[0]
    return {
        "parse":        best_parse,
        "score":        best_score,
        "root_ot":      root_info["root_ot"],
        "lemma":        root_info["lemma"],
        "dict_form":    root_info["dict_form"],
        "base_pos":     root_info["base_pos"],
        "surface_root": root_info["surface_root"],
        "tags":         tags,
    }


def lookup_word(word: str):
    if all(c.isdigit() or c in ".,-" for c in word) and any(c.isdigit() for c in word):
        return word.translate(str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")), "exact", word
    if word in lookup_dict:
        return lookup_dict[word], "exact", word
    lowered = lower_tr(word)
    if lowered in lookup_dict:
        return lookup_dict[lowered], "exact", lowered
    folded = fold_tr(word)
    if folded in lookup_folded:
        return lookup_folded[folded], "exact", lowered
    return None, None, None


def render_suffix_ottoman(tag: str, surface: str) -> str:
    normalized = normalize_surface_ascii(surface)
    if tag in {"FUTURE", "FUT_PART"} and normalized in {"yecek", "yeceğ"}:
        return {"yecek": "یەجك", "yeceğ": "یەجگ"}[normalized]
    if tag == "CAUSATIVE" and normalized == "t":
        return "ت"
    if tag == "PAST_PART" and normalized in {"diğ", "tiğ"}:
        return "دیگ"
    if tag == "CAUSATIVE" and normalized in {"dır", "dir", "dur", "dür", "tır", "tir", "tur", "tür"}:
        return render_ottoman(surface, historical=False)
    return render_ottoman(surface, historical=ENABLE_HISTORICAL_ORTHOGRAPHY)

# ============================================================
# §9  TAG → ALLOMORPH → OTTOMAN GENERATION
# ============================================================

def generate_from_tags(root_ot: str, tags: list[str], root_surface: str, lemma: str | None = None):
    normalized_tags = fuse_tag_strings(tags)
    if not validate_tag_sequence(normalized_tags):
        return None
    if normalized_tags == ["VERB", "A2SG"]:
        return {
            "surface_root": root_surface,
            "suffixes": [],
            "result": strip_infinitive_from_ottoman(root_ot),
        }

    morphs = build_underlying_morphs(normalized_tags)
    realized_root, realized_suffixes = realize_allomorphs(root_surface, morphs)

    rendered_suffixes = [
        {
            "tag":     part["tag"],
            "surface": part["surface"],
            "ottoman": render_suffix_ottoman(part["tag"], part["surface"]),
        }
        for part in realized_suffixes
    ]
    rendered_suffixes = adjust_past_person_rendering(rendered_suffixes)

    if normalized_tags and normalized_tags[0] == "VERB" and rendered_suffixes:
        stripped = strip_infinitive_from_ottoman(root_ot)
        if lower_tr(lemma or "") == "belirtmek":
            stripped = strip_infinitive_from_ottoman("بلیرتمك")
        if rendered_suffixes[0].get("tag") in NOMINAL_DERIVATION_TAGS:
            root_ot = stripped
        elif normalize_surface_ascii(realized_root) == normalize_surface_ascii(root_surface):
            root_ot = stripped
        else:
            alt = WORD_OVERRIDES.get(
                realized_root,
                render_ottoman(realized_root, historical=ENABLE_HISTORICAL_ORTHOGRAPHY),
            )
            root_ot = alt if alt != stripped else stripped
        aux_root_ot, rendered_suffixes = adjust_etmek_auxiliary_output(
            lemma, realized_root, rendered_suffixes
        )
        if aux_root_ot:
            root_ot = aux_root_ot
        elif (rendered_suffixes
                 and rendered_suffixes[0].get("tag") == "PASSIVE"
                 and normalize_surface_ascii(rendered_suffixes[0].get("surface", "")) == "n"
                 and normalize_surface_ascii(realized_root).endswith(("la", "le"))
                 and root_ot.endswith(("ه", "ە"))):
             # Passive -n on -la/-le verbs usually drops the final Ottoman vowel letter.
             root_ot = root_ot[:-1]
    elif (normalized_tags and normalized_tags[0] in {"NOUN", "ADJ", "PRON", "NUM"} and rendered_suffixes
            and lemma
            and normalize_surface_ascii(realized_root) != normalize_surface_ascii(lemma)):
        root_ot = adjust_softened_nominal_root_ottoman(root_ot, lemma, realized_root)

    return {
        "surface_root": realized_root,
        "suffixes":     rendered_suffixes,
        "result":       merge_ottoman(root_ot, rendered_suffixes),
    }


def generate_predicative_infinitive(selected: dict, analysis_word: str):
    morphemes = list(getattr(selected["parse"], "morphemes", []) or [])
    if "Inf1" not in morphemes or "Pres" not in morphemes or "Cop" not in morphemes:
        return None
    lemma = lower_tr(selected.get("lemma") or "")
    word  = lower_tr(analysis_word or "")
    if not lemma or not word.startswith(lemma):
        return None
    copula_surface = word[len(lemma):]
    if not copula_surface:
        return None
    copula_ottoman = render_ottoman(copula_surface, historical=ENABLE_HISTORICAL_ORTHOGRAPHY)
    return {
        "surface_root": lemma,
        "suffixes":     [{"tag": "COPULA_ASSERT", "surface": copula_surface, "ottoman": copula_ottoman}],
        "result":       selected["root_ot"] + copula_ottoman,
    }

def generate_participle_copula(selected: dict, analysis_word: str):
    morphemes = list(getattr(selected["parse"], "morphemes", []) or [])
    if "PresPart" not in morphemes or "Cop" not in morphemes:
        return None
    root_surface = selected.get("surface_root") or lower_tr(selected.get("lemma") or "")
    word = lower_tr(analysis_word or "")
    part_surface = ("y" if ends_with_vowel(root_surface) else "") + choose_harmony_A(root_surface) + "n"
    expected_prefix = root_surface + part_surface
    if not word.startswith(expected_prefix):
        return None
    copula_surface = word[len(expected_prefix):]
    if not copula_surface:
        return None
    root_ot = strip_infinitive_from_ottoman(selected["root_ot"])
    part_ottoman = OTTOMAN_SURFACE_OVERRIDES.get(part_surface, render_ottoman(part_surface, historical=ENABLE_HISTORICAL_ORTHOGRAPHY))
    if normalize_surface_ascii(copula_surface) in {"dır", "dir", "dur", "dür", "tır", "tir", "tur", "tür"}:
        copula_ottoman = "دیر"
    else:
        copula_ottoman = render_suffix_ottoman("COPULA_ASSERT", copula_surface)
    return {
        "surface_root": root_surface,
        "suffixes":     [
            {"tag": "PRES_PART", "surface": part_surface, "ottoman": part_ottoman},
            {"tag": "COPULA_ASSERT", "surface": copula_surface, "ottoman": copula_ottoman},
        ],
        "result":       root_ot + part_ottoman + copula_ottoman,
    }

def generate_present_participle(selected: dict, analysis_word: str):
    morphemes = list(getattr(selected["parse"], "morphemes", []) or [])
    if "PresPart" not in morphemes or "Cop" in morphemes:
        return None
    word = lower_tr(analysis_word or "")
    root_surface = selected.get("surface_root") or lower_tr(selected.get("lemma") or "")
    if not word.startswith(root_surface):
        return None
    part_surface = word[len(root_surface):]
    if not part_surface:
        return None
    root_ot = strip_infinitive_from_ottoman(selected["root_ot"])
    part_ottoman = OTTOMAN_SURFACE_OVERRIDES.get(part_surface, render_ottoman(part_surface, historical=ENABLE_HISTORICAL_ORTHOGRAPHY))
    return {
        "surface_root": root_surface,
        "suffixes":     [{"tag": "PRES_PART", "surface": part_surface, "ottoman": part_ottoman}],
        "result":       root_ot + part_ottoman,
    }

def generate_imperative_surface(selected: dict, analysis_word: str):
    morphemes = list(getattr(selected["parse"], "morphemes", []) or [])
    if "Imp" not in morphemes:
        return None
    root_surface = selected.get("surface_root") or lower_tr(selected.get("lemma") or "")
    word = lower_tr(analysis_word or "")
    if not word.startswith(root_surface):
        return None
    suffix_surface = word[len(root_surface):]
    if not suffix_surface:
        return None
    root_ot = strip_infinitive_from_ottoman(selected["root_ot"])
    imperative_surface_overrides = {
        "sın": "سین", "sin": "سین", "sun": "سین", "sün": "سین",
        "sınız": "سینز", "siniz": "سینز", "sunuz": "سینز", "sünüz": "سینز",
        "sınlar": "سینلر", "sinler": "سینلر", "sunlar": "سینلر", "sünler": "سینلر",
    }
    suffix_ottoman = imperative_surface_overrides.get(
        suffix_surface,
        OTTOMAN_SURFACE_OVERRIDES.get(
            suffix_surface,
            render_ottoman(suffix_surface, historical=ENABLE_HISTORICAL_ORTHOGRAPHY),
        ),
    )
    return {
        "surface_root": root_surface,
        "suffixes":     [{"tag": "IMP", "surface": suffix_surface, "ottoman": suffix_ottoman}],
        "result":       root_ot + suffix_ottoman,
    }


# ── All suffixes that may follow an apostrophe on a proper noun ────────────
# FIX: expanded from {ın/in/un/ün} to cover all case, possessive, and tense
# suffixes that attach to proper nouns in Turkish.
_APOSTROPHE_SUFFIXES: set[str] = {
    # Possessive
    "m", "n", "nın", "nin", "nun", "nün",
    "mız", "miz", "muz", "müz", "nız", "niz", "nuz", "nüz",
    "ım", "im", "um", "üm", "ın", "in", "un", "ün",
    "ımız", "imiz", "umuz", "ümüz", "ınız", "iniz", "unuz", "ünüz",
    "sı", "si", "su", "sü",
    "ları", "leri",
    # Accusative
    "ı", "i", "u", "ü", "yı", "yi", "yu", "yü",
    # Dative
    "a", "e", "ya", "ye",
    # Locative
    "da", "de", "ta", "te",
    # Ablative
    "dan", "den", "tan", "ten",
    # Genitive (classic)
    "nın", "nin", "nun", "nün",
    # Instrumental
    "la", "le", "yla", "yle",
    # Copula-assert
    "dır", "dir", "dur", "dür", "tır", "tir", "tur", "tür",
    # Copula-past
    "ydı", "ydi", "ydu", "ydü", "ydım", "ydim", "ydın", "ydin",
    "ydık", "ydik", "ydınız", "ydiniz",
    # Narrative
    "ymış", "ymiş",
    # Conditional
    "ysa", "yse",
    # Plural + case chains most commonly found on proper nouns
    "lar", "ler", "ların", "lerin", "lara", "lere",
    "lardan", "lerden", "larla", "lerle", "larda", "lerde",
}


def _render_apostrophe_suffix(suffix: str) -> str:
    """Render a post-apostrophe suffix with override table first, then render_ottoman."""
    ot = OTTOMAN_SURFACE_OVERRIDES.get(suffix)
    if ot:
        return ot
    return render_ottoman(suffix, historical=ENABLE_HISTORICAL_ORTHOGRAPHY)


def _render_numeric_apostrophe_suffix(suffix: str) -> str:
    """Render apostrophe suffixes attached to numerals."""
    normalized = lower_tr(suffix)
    numeric_suffix_overrides = {
        "da": "دە", "de": "دە", "ta": "دە", "te": "دە",
        "a": "ە", "e": "ە",
        "ya": "یە", "ye": "یە",
        "ı": "ی", "i": "ی", "u": "ی", "ü": "ی",
        "lık": "لیق", "lik": "ليك", "luk": "لوق", "lük": "لوك",
        "sı": "سی", "si": "سی", "su": "سی", "sü": "سی",
        "nı": "نی", "ni": "نی", "nu": "نی", "nü": "نی",
        "ını": "نی", "ini": "نی", "unu": "نی", "ünü": "نی",
        "sını": "سنی", "sini": "سنی", "sunu": "سنی", "sünü": "سنی",
        "ına": "نه", "ine": "نه", "una": "نه", "üne": "نه",
        "sına": "سنه", "sine": "سنه", "suna": "سنه", "süne": "سنه",
        "ından": "ندن", "inden": "ندن", "undan": "ندن", "ünden": "ندن",
        "sından": "سندن", "sinden": "سندن", "sundan": "سندن", "sünden": "سندن",
        "ıydı": "یدی", "iydi": "یدی", "uydu": "یدی", "üydü": "یدی",
        "sıydı": "سیدی", "siydi": "سیدی", "suydu": "سیدی", "süydü": "سیدی",
        "ındaydı": "ندەیدی", "indeydi": "یندەیدی", "undaydı": "ندەیدی", "ündeydi": "یندەیدی",
        "sındaydı": "سندەیدی", "sindeydi": "سیندەیدی", "sundaydı": "سندەیدی", "sündeydi": "سیندەیدی",
        "ındeymiş": "یندەیمش", "indeymiş": "یندەیمش", "undaymış": "ندەیمش", "ündeymiş": "یندەیمش",
        "sındendi": "سندەندی", "sindendi": "سیندەندی", "sundandı": "سندەندی", "sündendi": "سیندەندی",
    }
    ot = numeric_suffix_overrides.get(normalized)
    if ot:
        return ot
    return _render_apostrophe_suffix(normalized)


QUESTION_PARTICLE_SUFFIXES = (
    "mıyım", "miyim", "muyum", "müyüm",
    "mıyız", "miyiz", "muyuz", "müyüz",
    "mısın", "misin", "musun", "müsün",
    "mısınız", "misiniz", "musunuz", "müsünüz",
)


def render_concatenated_question_particle(token: str):
    word = lower_tr(token)
    for suffix in QUESTION_PARTICLE_SUFFIXES:
        if not word.endswith(suffix) or len(word) <= len(suffix):
            continue
        stem = token[:-len(suffix)]
        stem_ot, stem_source, _stem_debug = transliterate_token(stem)
        if stem_source == "missing":
            continue
        suffix_ot, suffix_source, _suffix_debug = transliterate_token(suffix)
        if suffix_source == "missing":
            continue
        return stem_ot + suffix_ot, stem, suffix
    return None

LEXICALIZED_NOMINAL_STEM_OVERRIDES: dict[str, str] = {
    "hayat":    "حیات",
    "kaybeden": "غائب ایدن",
}

LEXICALIZED_NOMINAL_SUFFIX_OVERRIDES: dict[str, str] = {
    "ı": "ی", "i": "ی", "u": "ی", "ü": "ی",
    "ını": "نی", "ini": "نی", "unu": "نی", "ünü": "نی",
    "ına": "نە", "ine": "نە", "una": "نە", "üne": "نە",
    "ından": "ندن", "inden": "ندن", "undan": "ندن", "ünden": "ندن",
    "lar": "لر", "ler": "لر",
    "lara": "لرە", "lere": "لرە",
}


def render_lexicalized_nominal_surface(token: str):
    word = lower_tr(token)
    for stem, stem_ot in sorted(
        LEXICALIZED_NOMINAL_STEM_OVERRIDES.items(),
        key=lambda item: len(item[0]),
        reverse=True,
    ):
        if not word.startswith(stem):
            continue
        suffix = word[len(stem):]
        if not suffix:
            return stem_ot, stem, ""
        suffix_ot = LEXICALIZED_NOMINAL_SUFFIX_OVERRIDES.get(
            suffix,
            OTTOMAN_SURFACE_OVERRIDES.get(
                suffix,
                render_ottoman(suffix, historical=ENABLE_HISTORICAL_ORTHOGRAPHY),
            ),
        )
        if suffix_ot:
            return stem_ot + suffix_ot, stem, suffix
    return None


def transliterate_token(token: str) -> tuple[str, str, str]:
    """Return (ottoman_string, source_label, debug_form) for one token."""

    analysis_word = token.replace("'", "").replace("\u2019", "")
    imperative_like_suffixes = (
        "sın", "sin", "sun", "sün",
        "sınız", "siniz", "sunuz", "sünüz",
        "sınlar", "sinler", "sunlar", "sünler",
    )

    # 1. Hand-written full-word override (case-sensitive first, then lowercased)
    if token in WORD_OVERRIDES:
        return WORD_OVERRIDES[token], "override", token
    lw = lower_tr(token)
    if lw == "belirt":
        return "بلیرت", "override", token
    if lw in WORD_OVERRIDES:
        return WORD_OVERRIDES[lw], "override", token

    # 2. Apostrophe split (proper noun + any case/possessive/tense suffix)
    #    FIX: handles all suffixes in _APOSTROPHE_SUFFIXES, not just genitive
    normalized = token.replace("\u2019", "'").replace("\u2018", "'")
    if "'" in normalized:
        base, suffix = normalized.split("'", 1)
        if base.isdigit() and suffix:
            base_ot = base.translate(str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩"))
            suf_ot = _render_numeric_apostrophe_suffix(suffix)
            return base_ot + suf_ot, "override", token
        base_result  = lookup_root_entry(base) or lookup_root_entry(lower_tr(base))
        if base_result and lower_tr(suffix) in _APOSTROPHE_SUFFIXES:
            base_ot = base_result[0]
            suf_ot  = _render_apostrophe_suffix(lower_tr(suffix))
            return base_ot + suf_ot, "override", token
        # If we cannot resolve the base, fall through to normal analysis

    # 2b. Inline numeral+suffix forms (e.g. 7lik -> ٧لك).
    numeric_inline_match = re.fullmatch(r"(\d+)([a-zA-ZçğıöşüÇĞİÖŞÜ]+)", normalized)
    if numeric_inline_match:
        base, suffix = numeric_inline_match.groups()
        suffix_lw = lower_tr(suffix)
        if suffix_lw in _APOSTROPHE_SUFFIXES or suffix_lw in {"lık", "lik", "luk", "lük"}:
            base_ot = base.translate(str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩"))
            suf_ot = _render_numeric_apostrophe_suffix(suffix_lw)
            return base_ot + suf_ot, "override", token

    # 2c. Recover colloquial tokens with a concatenated question particle.
    concatenated_question = render_concatenated_question_particle(token)
    if concatenated_question:
        ottoman, stem, suffix = concatenated_question
        return ottoman, "surface", f"{stem} + {suffix}"

    lexicalized_nominal = render_lexicalized_nominal_surface(token)
    if lexicalized_nominal:
        ottoman, stem, suffix = lexicalized_nominal
        return ottoman, "surface", f"{stem} + {suffix}" if suffix else stem

    # 3. Prefer valid imperative parses over dictionary matches for imperative-like surfaces.
    selected = None
    if lw.endswith(imperative_like_suffixes):
        selected = select_parse_for_generation(analysis_word)
        if selected:
            imperative = generate_imperative_surface(selected, analysis_word)
            if imperative:
                debug = f"{selected['lemma']} :: IMPERATIVE_SURFACE :: {analysis_word}"
                return imperative["result"], "tags", debug

    # 4. Surface-aware fallback for unresolved verb suffix chains.
    surface_fallback = generate_surface_verb_fallback(token)
    if surface_fallback:
        ottoman, stem, suffix = surface_fallback
        return ottoman, "surface", f"{stem} + {suffix}"
    future_past_fallback = generate_surface_future_chain_fallback(token)
    if future_past_fallback:
        ottoman, stem, suffix = future_past_fallback
        return ottoman, "surface", f"{stem} + {suffix}"

    # 5. Direct dictionary / number lookup
    direct_ot, direct_src, direct_form = lookup_word(token)
    if direct_ot is not None:
        return direct_ot, direct_src, direct_form

    # 6. English / foreign-script detection
    #    Checked before Zeyrek: Zeyrek cannot parse Latin loanwords correctly.
    if is_likely_english(token):
        eng_ot = render_english_ottoman(token)
        return eng_ot, "english", token

    # 7. Morphological analysis via Zeyrek
    if selected is None:
        selected = select_parse_for_generation(analysis_word)
    if selected:
        imperative = generate_imperative_surface(selected, analysis_word)
        if imperative:
            debug = f"{selected['lemma']} :: IMPERATIVE_SURFACE :: {analysis_word}"
            return imperative["result"], "tags", debug

        pres_part = generate_present_participle(selected, analysis_word)
        if pres_part:
            debug = f"{selected['lemma']} :: PRESENT_PARTICIPLE :: {analysis_word}"
            return pres_part["result"], "tags", debug

        part_cop = generate_participle_copula(selected, analysis_word)
        if part_cop:
            debug = f"{selected['lemma']} :: PARTICIPLE_COPULA :: {analysis_word}"
            return part_cop["result"], "tags", debug

        pred_inf = generate_predicative_infinitive(selected, analysis_word)
        if pred_inf:
            debug = f"{selected['lemma']} :: PREDICATIVE_INF :: {analysis_word}"
            return pred_inf["result"], "tags", debug

        generated = generate_from_tags(
            selected["root_ot"], selected["tags"], selected["surface_root"],
            lemma=selected.get("lemma"),
        )
        if generated:
            suffix_str = " + ".join(p["surface"] for p in generated["suffixes"])
            debug = (
                f"{selected['lemma']} :: "
                f"{'+'.join(selected['tags'])} :: "
                f"{selected['surface_root']}+{suffix_str}"
            )
            return generated["result"], "tags", debug

    # 8. Fallback: auto-render without deep morphological analysis
    parses = flatten_analyses(analysis_word)
    if parses:
        pos = str(getattr(parses[0], "pos", ""))
        if "Prop" not in pos and "Abbrv" not in pos and "Unk" not in pos:
            return render_ottoman(token, historical=False), "auto", token

    # 9. Unresolved – bracket for review
    return f"[{token}]", "missing", token

# ============================================================
# §10  SHARED TOKENIZER + TEST SENTENCE PIPELINE
#      FIX: tokenization logic extracted into tokenize_and_transliterate
#      so both the display path and the batch path share one implementation.
# ============================================================

def tokenize_and_transliterate(line: str):
    """Yield (token, ottoman, source, debug_form) for each token in `line`.

    Non-word spans between tokens are yielded with source='whitespace' so
    callers can reconstruct the original spacing without re-scanning the line.
    """
    cursor = 0
    for match in _TOKEN_RE.finditer(line):
        start, end = match.span()
        # Preserve inter-token whitespace / characters
        if start > cursor:
            gap = line[cursor:start]
            yield gap, gap, "whitespace", gap

        token = match.group(0)
        if not is_word_token(token):
            ot = convert_ottoman_punctuation(token)
            yield token, ot, "punct", token
        else:
            ot, src, dbg = transliterate_token(token)
            yield token, ot, src, dbg

        cursor = end

    if cursor < len(line):
        tail = line[cursor:]
        yield tail, tail, "whitespace", tail


def process_test_sentence(line: str) -> None:
    """Run pipeline on a single sentence, print a token table and render HTML."""
    ot_parts: list[str] = []

    for token, ot, source, debug_form in tokenize_and_transliterate(line):
        ot_parts.append(ot)
        if source not in ("whitespace",):
            print(
                f"Kelime: {token:<20} | "
                f"Kaynak: {source:<11} | "
                f"Sozluk: {str(debug_form):<70} | "
                f"Sonuc: {ot}"
            )

    ottoman_line = "".join(ot_parts)
    display(HTML(
        "<div style='font-size:38px;direction:ltr;font-family:serif;"
        "border:1px solid #ccc;padding:14px;margin-bottom:6px;'>"
        + line + "</div>"
        + "<div style='font-size:38px;direction:rtl;font-family:serif;"
        "border:1px solid #ccc;padding:14px;'>"
        + ottoman_line + "</div>"
    ))


# ── Run on the test sentence ─────────────────────────────────────────────────
process_test_sentence(TEST_SENTENCE)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Sezer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Kelime: vefat                | Kaynak: exact       | Sozluk: vefat                                                                  | Sonuc: وفات
Kelime: eden                 | Kaynak: exact       | Sozluk: eden                                                                   | Sonuc: ایدن
Kelime: ve                   | Kaynak: exact       | Sozluk: ve                                                                     | Sonuc: و
Kelime: yitirilenlere        | Kaynak: tags        | Sozluk: yitirmek :: PRESENT_PARTICIPLE :: yitirilenlere                        | Sonuc: ييتيرایلهنلهره
Kelime: Allah'tan            | Kaynak: override    | Sozluk: Allah'tan                                                              | Sonuc: اللّٰهدن
Kelime: rahmet               | Kaynak: override    | Sozluk: rahmet                                                                 | Sonuc: رحمت
Kelime: ,                    | Kaynak: punct       | Sozluk: ,                                                   

## Hücre 2: Toplu İşlem Boru Hattı

Paralel işleme (`BATCH_WORKERS`), periyodik flush (`FLUSH_INTERVAL`), güvenli güven yönlendirmesi ve `$OTTOMAN_BASE_DIR` env-var desteği içerir.

In [2]:
# ============================================================
# §11  BATCH PIPELINE
#      Fixes:
#       • resolve_io_paths: hardcoded Windows path removed;
#         replaced with env-var BASE_DIR override.
#       • Confidence routing: single source-of-truth ("missing" in
#         sources) instead of mixing bracket-check with list-check.
#       • Periodic flush every FLUSH_INTERVAL lines (default 500)
#         so output is safe even if kernel dies mid-run.
#       • Optional parallelism via ThreadPoolExecutor (BATCH_WORKERS).
#       • MAX_LINES=None means process the full file.
# ============================================================

import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed


def process_line(line: str):
    line = line.strip()
    if not line:
        return None

    ot_parts: list[str] = []
    sources:  list[str] = []

    for _token, ot, source, _dbg in tokenize_and_transliterate(line):
        ot_parts.append(ot)
        if source not in ("whitespace",):
            sources.append(source)

    ottoman_line = "".join(ot_parts)
    scores       = [SCORE_MAP.get(s, 0.0) for s in sources]
    confidence   = sum(scores) / len(scores) if scores else 1.0

    result_data = {
        "turkish":    line,
        "ottoman":    ottoman_line,
        "confidence": round(confidence, 3),
        "sources":    sources,
    }

    # FIX: single source-of-truth: route on sources list only.
    # Bracket check ("["in ottoman_line) is NOT used here because future
    # code may emit brackets for non-missing reasons.
    if "missing" in sources:
        target_file = OUT_MISSING
    elif confidence == 1.0:
        target_file = OUT_OK
    else:
        target_file = OUT_APPROX

    return target_file, json.dumps(result_data, ensure_ascii=False)


def resolve_io_paths() -> tuple[Path, dict]:
    """Find the input file.

    Search order:
      1. $OTTOMAN_BASE_DIR env var (recommended for Colab / server use)
      2. Current working directory
      3. <cwd>/Ottoman_Project/scripts
    """
    # FIX: the hardcoded Windows path has been removed.
    base_override = os.environ.get("OTTOMAN_BASE_DIR")
    candidates = []
    if base_override:
        candidates.append(Path(base_override))
    candidates += [
        Path.cwd(),
        Path.cwd() / "Ottoman_Project" / "scripts",
    ]

    for base_dir in candidates:
        input_path = base_dir / INPUT_FILE
        if input_path.exists():
            return input_path, {
                OUT_OK:      base_dir / OUT_OK,
                OUT_APPROX:  base_dir / OUT_APPROX,
                OUT_MISSING: base_dir / OUT_MISSING,
            }

    raise FileNotFoundError(
        f"Input file '{INPUT_FILE}' not found. "
        "Set the OTTOMAN_BASE_DIR environment variable to the directory "
        "that contains it."
    )


# ── Run ──────────────────────────────────────────────────────────────────────
input_path, output_paths = resolve_io_paths()
counts    = {OUT_OK: 0, OUT_APPROX: 0, OUT_MISSING: 0}
processed = 0
skipped   = 0

with (
    input_path.open("r", encoding="utf-8") as src,
    output_paths[OUT_OK].open("w",      encoding="utf-8") as ok_dst,
    output_paths[OUT_APPROX].open("w",  encoding="utf-8") as approx_dst,
    output_paths[OUT_MISSING].open("w", encoding="utf-8") as missing_dst,
):
    writers = {
        OUT_OK:      ok_dst,
        OUT_APPROX:  approx_dst,
        OUT_MISSING: missing_dst,
    }

    # Read lines up to MAX_LINES (None = all)
    raw_lines: list[str] = []
    for line_no, raw in enumerate(src, start=1):
        if MAX_LINES is not None and line_no > MAX_LINES:
            break
        raw_lines.append(raw)

    if BATCH_WORKERS > 1:
        # ── Parallel path ────────────────────────────────────────────────
        with ThreadPoolExecutor(max_workers=BATCH_WORKERS) as pool:
            futures = {pool.submit(process_line, ln): i for i, ln in enumerate(raw_lines)}
            for future in as_completed(futures):
                result = future.result()
                if result is None:
                    skipped += 1
                    continue
                target_file, json_str = result
                writers[target_file].write(json_str + "\n")
                counts[target_file] += 1
                processed += 1
                if processed % FLUSH_INTERVAL == 0:
                    for w in writers.values():
                        w.flush()
                    print(f"  … {processed:,} lines processed "
                          f"(ok={counts[OUT_OK]}, "
                          f"approx={counts[OUT_APPROX]}, "
                          f"missing={counts[OUT_MISSING]})")
    else:
        # ── Sequential path ──────────────────────────────────────────────
        for raw in raw_lines:
            result = process_line(raw)
            if result is None:
                skipped += 1
                continue
            target_file, json_str = result
            writers[target_file].write(json_str + "\n")
            counts[target_file] += 1
            processed += 1
            if processed % FLUSH_INTERVAL == 0:
                for w in writers.values():
                    w.flush()
                print(f"  … {processed:,} lines processed "
                      f"(ok={counts[OUT_OK]}, "
                      f"approx={counts[OUT_APPROX]}, "
                      f"missing={counts[OUT_MISSING]})")

    # Final flush
    for w in writers.values():
        w.flush()

print(
    f"\n✓ Done. {processed:,} lines written "
    f"({skipped} blank skipped).\n"
    f"  ✔ Exact  : {counts[OUT_OK]:>7,}  → {output_paths[OUT_OK].name}\n"
    f"  ~ Approx : {counts[OUT_APPROX]:>7,}  → {output_paths[OUT_APPROX].name}\n"
    f"  ✗ Missing: {counts[OUT_MISSING]:>7,}  → {output_paths[OUT_MISSING].name}"
)

  … 500 lines processed (ok=234, approx=136, missing=130)
  … 1,000 lines processed (ok=493, approx=244, missing=263)
  … 1,500 lines processed (ok=716, approx=370, missing=414)
  … 2,000 lines processed (ok=953, approx=497, missing=550)
  … 2,500 lines processed (ok=1214, approx=621, missing=665)
  … 3,000 lines processed (ok=1468, approx=748, missing=784)
  … 3,500 lines processed (ok=1724, approx=862, missing=914)
  … 4,000 lines processed (ok=1976, approx=982, missing=1042)
  … 4,500 lines processed (ok=2230, approx=1105, missing=1165)
  … 5,000 lines processed (ok=2451, approx=1218, missing=1331)
  … 5,500 lines processed (ok=2676, approx=1329, missing=1495)
  … 6,000 lines processed (ok=2898, approx=1450, missing=1652)
  … 6,500 lines processed (ok=3125, approx=1571, missing=1804)
  … 7,000 lines processed (ok=3354, approx=1691, missing=1955)
  … 7,500 lines processed (ok=3591, approx=1792, missing=2117)
  … 8,000 lines processed (ok=3820, approx=1920, missing=2260)
  … 8,500 lin

In [3]:
import random
import json

output_file_path = output_paths[OUT_OK]  # Using the exact matches file

# Read all lines from the file
with open(output_file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Randomly select 20 lines (or fewer if the file has less than 20)
num_samples = min(20, len(lines))
random_samples = random.sample(lines, num_samples)

print(f"Displaying {num_samples} random translations from {output_file_path.name}:\n")

for i, line in enumerate(random_samples):
    data = json.loads(line)
    turkish_text = data.get('turkish', 'N/A')
    ottoman_text = data.get('ottoman', 'N/A')
    confidence = data.get('confidence', 'N/A')

    print(f"--- Translation {i+1} ---")
    print(f"Turkish   : {turkish_text}")
    print(f"Ottoman   : {ottoman_text}")
    print(f"Confidence: {confidence}\n")

Displaying 20 random translations from fsmtamam.jsonl:

--- Translation 1 ---
Turkish   : Okyanus’u kim öldürdü?
Ottoman   : اوقيانوسی كیم اولدیردی؟
Confidence: 1.0

--- Translation 2 ---
Turkish   : Polis, Yunanca şehir anlamına gelmektedir.
Ottoman   : پولیس، يونانجه شهير آڭلامنە گلمكدەدر.
Confidence: 1.0

--- Translation 3 ---
Turkish   : Suçlanan üç kişiye görüşlerini almak için ulaşmak mümkün olmadı.
Ottoman   : صوچلانان أوچ كیشی یە گوروشلرینی آلمق ایچون اولاشمق ممكن اولمدی.
Confidence: 1.0

--- Translation 4 ---
Turkish   : (Ben bir gün başarılı bir yönetici olacağım.)
Ottoman   : (بن بر گون باشاريلي بر يوڭتيجي اولاجغم.)
Confidence: 1.0

--- Translation 5 ---
Turkish   : Eğer kocama yazmayı
Ottoman   : اگر قوجامه يازمەیی
Confidence: 1.0

--- Translation 6 ---
Turkish   : Plastik kirliliği: Okyanuslar gerçekten insan eliyle temizlenebilir mi? →
Ottoman   : پلاستيك قيرلیلقی: اوقيانوسلر گرچكدن انسان اليله تميزلنەبیل مي؟ →
Confidence: 1.0

--- Translation 7 ---
Turkish   : Kasa Çok A